# OriginSib-Comp v2 — Haplotype-resolved cis-epistasis and regulatory compensation (genome-wide, MAGE ASE + GEUVADIS)

**What changes from v1:** v1 tested interactions in *total* expression. Between-person noise (environment, trans genetics, batch) left it under-powered: ΔR² vs PrediXcan-style EN was −0.008. v2 tests the core question directly with **haplotype-level allele-specific expression (ASE)**. Each person's two haplotypes are measured in the same cell, so each person is their own control.

For a gene with haplotype expressions $e_A, e_B$, the log allelic ratio is $\log(a/b) = \log e_A - \log e_B$. Under a log-additive haplotype model (aFC-n):
$$\text{logit}\,p = \sum_j \beta_j (h_{Aj}-h_{Bj}) + \gamma_{jk}\,(h_{Aj}h_{Ak}-h_{Bj}h_{Bk})$$
$\gamma_{jk}\neq 0$ means that variants $j,k$ **on the same chromosome copy** act non-additively (**cis-epistasis**). If $\beta_j\beta_k<0$ and the joint haplotype returns toward the reference level, the pair is a **compensation** candidate. Interactions *between* the two parental copies (trans-haplotype) cancel in an allelic ratio. Those are tested in v1 (total expression).

| Mode | What it does | Run time (T4 + 4 CPU, estimate) |
|---|---|---|
| `PLAN` | Writes `preregistration.md` (hypotheses, models, thresholds, config hash) **before any analysis**. Post it on OSF. | 1 min |
| `BATCH` (1–4) | Downloads the data for the batch's chromosomes, checks that ASE phase is aligned with the 1000G 30× haplotypes, runs the discovery → replication cis-epistasis scan, negative controls, held-out-individual prediction, GEUVADIS cross-assay replication, and the **targeted test of Siraj et al. 2026 MPRA variant pairs** (Zenodo 15297965, auto-downloaded) | ~2–3 h per batch |
| `COMBINE` | Genome-wide BH-FDR, evidence levels, compensation & cCRE enrichment, **Gap A MPRA-transfer test**, final Tables/Figures. Also ingests v1 results if attached | 5–10 min |

### How to run on Kaggle
1. Internet **ON**. A GPU is not needed (CPU statistics).
2. `MODE="PLAN"` → Run All → post `preregistration.md` on OSF.
3. `MODE="BATCH"`, `BATCH=1` → **Save & Run All (Commit)**. Repeat for `BATCH=2,3,4`, one commit each.
4. Make a new version with `MODE="COMBINE"`. Add the four batch outputs as inputs: *Add Input → Your Work → this notebook's versions*. Optionally add the v1 output too. Then Run All.

**Claims policy:** L1 tested → L2 discovery FDR → L3 replicated in independent individuals → **L4** L3 + MPRA support (attach your MPRA table) or **L4\*** L3 + ENCODE cCRE annotation only → **L5** L3 + perturbation validation (attach your CRISPR / base-editing table). L4/L5 are assigned **only** from experimental tables you attach. The notebook never creates them.

**Attaching wet-lab results (optional, for COMBINE):** add a CSV as Kaggle input with columns `chrom,pos1,pos2` plus either `y00,y10,y01,y11[,se]` / `eps[,eps_se]` (MPRA) or `perturb_effect,perturb_p` (perturbation). Positions are GRCh38. Pair order does not matter.

In [ ]:
# =====================================================================
# 0. CONFIGURATION
# =====================================================================
import os, sys, re, gc, io, json, math, time, gzip, glob, random, shutil, hashlib, tarfile, zipfile
import platform, warnings, itertools, subprocess, bisect, inspect, traceback
import urllib.request, urllib.error
from pathlib import Path
from collections import defaultdict, OrderedDict
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

CFG = dict(
    MODE              = "BATCH",      # "PLAN" | "BATCH" | "COMBINE"
    BATCH             = 1,            # 1..4 (chromosome groups below, ~25% of the genome each)
    BATCHES           = {1: ["chr1", "chr2", "chr3", "chr21"],
                         2: ["chr4", "chr5", "chr6", "chr7", "chr22"],
                         3: ["chr8", "chr9", "chr10", "chr11", "chr12"],
                         4: ["chr13", "chr14", "chr15", "chr16", "chr17", "chr18", "chr19", "chr20"]},
    CHROMS_OVERRIDE   = None,         # e.g. ["chr22"] for a quick pilot
    SELF_TEST         = False,        # synthetic offline test. NEVER report these numbers.
    CIS_WINDOW        = 100_000,
    MAF_MIN           = 0.05,         # in the ASE cohort
    MIN_COUNT         = 8,            # min total haplotypic reads for an individual-gene observation
    MIN_IND           = 50,           # min individuals with MIN_COUNT reads in the discovery half
    K_PAIR            = 8,            # candidate variants per gene for pair tests (28 pairs)
    K_ADD             = 16,           # additive variants for the prediction model
    N_COND            = 10,           # conditional additive variants in every pair model (LD / tagging control)
    CLUMP_R2          = 0.80,
    PAIR_MAX_R2       = 0.50,
    MIN_INFORMATIVE   = 15,           # min individuals with a non-zero cis-pair regressor
    N_RANDOM_PAIRS    = 8,
    N_PERM            = 2,            # orientation-flip permutations (null of the whole pipeline)
    FDR               = 0.05,
    REP_P             = 0.05,         # one-sided replication threshold
    GEU_TEST_P        = 1e-3,         # discovery p below which pairs are tested in GEUVADIS-only individuals
    N_FOLDS           = 5,
    BOOTSTRAP         = 2000,
    MAX_GENES         = None,
    USE_CCRE          = True,
    USE_GEUVADIS      = True,
    USE_MPRA_TARGETED = True,         # Gap A: test Siraj et al. (Nature 2026) MPRA variant pairs on natural haplotypes
    MPRA_CELL         = "gm12878",    # LCL, same cell type as MAGE/GEUVADIS (secondary MPRA scope)
    MPRA_PRIMARY      = "all",        # primary MPRA significance: random-effects over ALL cell types & windows
    MPRA_MIN_DNA      = 10,           # min mean plasmid count of all four constructs
    TARGET_MAX_R2     = 0.90,         # pairs in stronger LD cannot separate gamma from main effects
    N_JOBS            = -1,
    ALIGN_MIN_FRAC    = 0.02,         # min fraction of genes with a strong signed haplotype-ASE association
)
IS_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
OUT  = WORK / "originsib_v2_results"
DATA = Path("/tmp/originsib_v2_data")
for d in [OUT, OUT / "tables", OUT / "figures", DATA]:
    d.mkdir(parents=True, exist_ok=True)
if CFG["SELF_TEST"]:
    CFG.update(CHROMS_OVERRIDE=["chr22"], N_PERM=1, BOOTSTRAP=300, USE_GEUVADIS=True, MIN_IND=30)
CHROMS = CFG["CHROMS_OVERRIDE"] or CFG["BATCHES"][CFG["BATCH"]]
T0 = time.time(); LOG = []
def log(*a):
    s = " ".join(str(x) for x in a); print(s, flush=True); LOG.append(f"[{time.time()-T0:8.1f}s] {s}")
log(f"OriginSib-Comp v2 | MODE={CFG['MODE']} | BATCH={CFG['BATCH']} | CHROMS={CHROMS} | SELF_TEST={CFG['SELF_TEST']}")

import numpy as np, pandas as pd
from scipy import stats
from scipy.stats import norm, chi2, spearmanr, wilcoxon, fisher_exact, rankdata
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import GroupKFold
from joblib import Parallel, delayed
N_JOBS = os.cpu_count() if CFG["N_JOBS"] in (-1, None) else CFG["N_JOBS"]
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold"})
PAL = ["#2F5D8A", "#C0504D", "#4E9A6B", "#8064A2", "#E09A3E", "#4BACC6", "#7F7F7F"]
VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "statsmodels": sm.__version__ if hasattr(sm, "__version__") else "", "cpus": os.cpu_count()}
def md(df, index=False):
    try: return df.to_markdown(index=index)
    except Exception: return df.to_string(index=index)
def save_table(df, name, index=False):
    df.to_csv(OUT / "tables" / f"{name}.csv", index=index)
    with open(OUT / "tables" / f"{name}.md", "w") as f: f.write(md(df, index))
    return df
def savefig(fig, name):
    for ext in ("png", "pdf"): fig.savefig(OUT / "figures" / f"{name}.{ext}", bbox_inches="tight")
    plt.show(); plt.close(fig)
def cfg_hash():
    keep = {k: v for k, v in CFG.items() if k not in ("MODE", "BATCH", "CHROMS_OVERRIDE")}
    return hashlib.sha256(json.dumps(keep, sort_keys=True, default=str).encode()).hexdigest()[:16]

## 1. Pre-registration (MODE = "PLAN")

In [ ]:
PREREG = f"""# OriginSib-Comp v2 — pre-registered analysis plan
Generated: {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())}  |  config hash: `{cfg_hash()}`

## Hypotheses
H1 (cis-epistasis): for some cis-regulatory variant pairs (j,k), the effect on haplotype expression is non-additive on the
log scale: gamma_jk != 0 in logit p = sum_j beta_j (hA_j - hB_j) + gamma_jk (hA_j hA_k - hB_j hB_k), p = a/(a+b).
H2 (compensation): among replicated interactions, opposing main effects (beta_j*beta_k < 0) with a joint haplotype effect
closer to the reference than either single-variant effect occur more often than in matched random pairs.
H3 (prediction): adding sparse cis-pair terms improves held-out prediction of allelic log-ratios over the additive
haplotype model (weighted R^2), with the gain lost when cis configurations are phase-shuffled.
H4 (MPRA transfer, Gap A): variant pairs with non-additive MPRA activity (Siraj et al., Nature 2026; DerSimonian-Laird
random-effects meta over oligo windows; PRIMARY scope = all cell types, SECONDARY = GM12878 only; FDR < 5%) show
natural-haplotype cis-epistasis (gamma) more often, and with the same sign more often, than MPRA-additive pairs (p > 0.5). Primary tests: Fisher test of natural p<0.05 (MPRA-FDR vs MPRA-null pairs);
binomial test of sign concordance among MPRA-FDR pairs; Spearman(eps_MPRA, gamma_natural).

## Data
MAGE haplotype-level ASE (phASER; 731 LCLs, 26 populations; PejLab/ASE_Data); 1000 Genomes 30x phased genotypes
(GRCh38); GEUVADIS total expression (cross-assay replication in individuals not in MAGE); ENCODE SCREEN cCREs; GENCODE v38;
Siraj et al. 2026 MPRA normalized counts (Zenodo 15297965, data_MPRA.zip): 2x2 variant-pair constructs (focal x neighbour,
ref/alt), interaction eps = y11 - y10 - y01 + y00 on log2(RNA/DNA), replicate SE, random-effects meta per pair.

## Fixed analysis choices (not to be changed after seeing results)
- cis window +/-{CFG['CIS_WINDOW']//1000} kb of the TSS; MAF >= {CFG['MAF_MIN']}; observation if haplotypic reads >= {CFG['MIN_COUNT']};
  gene tested if >= {CFG['MIN_IND']} such individuals in the discovery half.
- Individuals split 50/50 into discovery/replication by family, stratified by super-population.
- Candidates: top {CFG['K_PAIR']} variants by discovery-half marginal haplotype-ASE association, LD-clumped (r^2 < {CFG['PAIR_MAX_R2']});
  all pairs tested if >= {CFG['MIN_INFORMATIVE']} individuals have a non-zero cis-pair regressor; {CFG['N_COND']} conditional variants.
- Model: binomial GLM (logit) with robust sandwich (HC0) standard errors. Primary test: gamma_jk (Wald).
- Discovery: genome-wide BH-FDR {CFG['FDR']}. Replication: same sign, one-sided p < {CFG['REP_P']} in the replication half.
- Negative controls: allelic-orientation flips (permutation null), MAF-matched random pairs, phase-shuffled cis features.
- Prediction: {CFG['N_FOLDS']}-fold family-grouped CV; additive ridge vs additive + Lasso cis-pairs with the 1-SE rule;
  metric = weighted R^2 of empirical allelic log-ratio; block bootstrap 95% CI; Wilcoxon over genes.
- Evidence levels: L2 discovery FDR; L3 replicated; L4* replicated + ENCODE cCRE (annotation, not perturbation).
- Total-expression v1 analysis (trans-haplotype, PrediXcan benchmark) is reported as a separate, already-completed analysis.

## Full configuration
```json
{json.dumps(CFG, indent=1, default=str)}
```
"""
with open(OUT / "preregistration.md", "w") as f: f.write(PREREG)
log("Pre-registration written:", OUT / "preregistration.md")
if CFG["MODE"] == "PLAN":
    print(PREREG)
    raise SystemExit("MODE=PLAN: pre-registration written. Post it (e.g. OSF), then set MODE='BATCH'.")

In [ ]:
# =====================================================================
# 0c. ROBUST DOWNLOADER (shared with v1)
# =====================================================================
UA = {"User-Agent": "Mozilla/5.0 (research-notebook; python-urllib)"}
MANIFEST = []

def _md5(path, nbytes=None):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(1 << 22)
            if not b: break
            h.update(b)
    return h.hexdigest()

def find_in_kaggle_input(fname):
    roots = [Path("/kaggle/input")]
    base = fname[:-3] if fname.endswith(".gz") else fname
    for root in roots:
        if not root.exists(): continue
        for dp, dn, fn in os.walk(root):
            for f in fn:
                if f == fname or f == base:
                    return Path(dp) / f
    return None

def _http_get(url, dest, timeout=90):
    tmp = Path(str(dest) + ".part")
    pos = tmp.stat().st_size if tmp.exists() else 0
    req = urllib.request.Request(url, headers=UA)
    if pos and url.startswith("http"):
        req.add_header("Range", f"bytes={pos}-")
    with urllib.request.urlopen(req, timeout=timeout) as r:
        code = getattr(r, "status", 200)
        if pos and code != 206: pos = 0
        cl = r.headers.get("Content-Length")
        total = (int(cl) + pos) if cl else None
        done, last = pos, time.time()
        with open(tmp, "ab" if pos else "wb") as f:
            while True:
                chunk = r.read(1 << 20)
                if not chunk: break
                f.write(chunk); done += len(chunk)
                if time.time() - last > 20:
                    msg = f"{done/1e6:,.0f} MB" + (f" / {total/1e6:,.0f} MB" if total else "")
                    print("   ...", dest.name, msg, flush=True); last = time.time()
    if total and done < total:
        raise IOError(f"incomplete download {done}/{total}")
    tmp.rename(dest)

def _valid(path, min_bytes):
    if not path.exists() or path.stat().st_size < min_bytes: return False
    if str(path).endswith(".gz"):
        try:
            with gzip.open(path, "rb") as f: f.read(4096)
        except Exception:
            return False
    return True

def fetch(fname, urls, min_bytes=1000, required=True, desc="", resolvers=()):
    """cache -> /kaggle/input -> each URL (resume + retries; a 403/404/410 is not retried) -> resolver functions
       that discover new URLs at run time (e.g. crawling the EBI directory listing)."""
    dest = DATA / fname
    if _valid(dest, min_bytes):
        MANIFEST.append(dict(file=fname, source="cache", path=str(dest), bytes=dest.stat().st_size)); return dest
    k = find_in_kaggle_input(fname)
    if k is not None:
        log(f"[data] {fname}: found in Kaggle input -> {k}")
        MANIFEST.append(dict(file=fname, source="kaggle_input", path=str(k), bytes=k.stat().st_size)); return k
    if CFG["SELF_TEST"]:
        raise FileNotFoundError(f"SELF_TEST: synthetic file missing {fname}")
    errs, tried = [], set()
    def _try(url):
        for attempt in range(3):
            try:
                log(f"[data] downloading {fname} ({desc}) <- {url}  [try {attempt+1}]")
                _http_get(url, dest)
                if _valid(dest, min_bytes):
                    MANIFEST.append(dict(file=fname, source=url, path=str(dest), bytes=dest.stat().st_size,
                                         md5=_md5(dest)))
                    log(f"[data] OK {fname}: {dest.stat().st_size/1e6:,.1f} MB")
                    return True
                raise IOError("downloaded file failed validation")
            except Exception as e:
                errs.append(f"{url}: {type(e).__name__}: {e}")
                if dest.exists() and not _valid(dest, min_bytes): dest.unlink()
                if isinstance(e, urllib.error.HTTPError) and e.code in (400, 401, 403, 404, 410):
                    return False          # permanent: do not retry this URL
                time.sleep(3 * (attempt + 1))
        return False
    def _run(lst):
        for url in lst:
            if url and url not in tried:
                tried.add(url)
                if _try(url): return True
        return False
    if _run(urls): return dest
    for res in resolvers:
        try:
            found = res()
        except Exception as e:
            errs.append(f"resolver {getattr(res, '__name__', res)}: {type(e).__name__}: {e}"); found = []
        if found: log(f"[data] resolver found candidate URLs for {fname}: {found[:4]}")
        if _run(found): return dest
    msg = (f"Could not obtain {fname}. Kaggle: turn Settings->Internet ON, or attach the file "
           f"'{fname}' as a Kaggle Dataset. Errors:\n  " + "\n  ".join(errs[-8:]))
    if required: raise RuntimeError(msg)
    log("[data][optional-missing]", msg.splitlines()[0]); return None

def crawl_listing(bases, exact, pattern=None, max_depth=3, max_pages=80):
    """Breadth-first search of HTTP directory listings (EBI FTP over HTTPS) for a file.
       Returns URLs: exact-name matches first, then files matching `pattern`."""
    hits_exact, hits_pat, seen, pages = [], [], set(), 0
    queue = [(b if b.endswith("/") else b + "/", 0) for b in bases]
    while queue and pages < max_pages:
        url, d = queue.pop(0)
        if url in seen: continue
        seen.add(url); pages += 1
        try:
            req = urllib.request.Request(url, headers=UA)
            with urllib.request.urlopen(req, timeout=60) as r:
                html = r.read().decode("utf-8", "ignore")
        except Exception:
            continue
        for href in re.findall(r'href="([^"?#]+)"', html):
            if href.startswith(("/", "http", "..", "mailto")) or href in ("./",): continue
            full = url + href
            name = href.rstrip("/").split("/")[-1]
            if href.endswith("/"):
                if d + 1 <= max_depth: queue.append((full, d + 1))
            elif name == exact:
                hits_exact.append(full)
            elif pattern and re.search(pattern, name):
                hits_pat.append(full)
        if hits_exact: break
    return hits_exact + hits_pat

def http_json(url, timeout=60):
    req = urllib.request.Request(url, headers=dict(UA, Accept="application/json"))
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())

def http_post_json(url, payload, timeout=120):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers=dict(UA, **{"Content-Type": "application/json", "Accept": "application/json"}))
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())

G1K = ["https://ftp.1000genomes.ebi.ac.uk/vol1/ftp", "http://ftp.1000genomes.ebi.ac.uk/vol1/ftp",
       "https://ftp-trace.ncbi.nlm.nih.gov/1000genomes/ftp", "https://ftp-trace.ncbi.nih.gov/1000genomes/ftp",
       "ftp://ftp.1000genomes.ebi.ac.uk/vol1/ftp"]
HC = "data_collections/1000G_2504_high_coverage"
PED_NAME = "20130606_g1k_3202_samples_ped_population.txt"
PED_URLS = [f"{b}/{HC}/{PED_NAME}" for b in G1K]
def vcf_name(ch): return f"1kGP_high_coverage_Illumina.{ch}.filtered.SNV_INDEL_SV_phased_panel.vcf.gz"
def vcf_urls(ch): return [f"{b}/{HC}/working/20220422_3202_phased_SNV_INDEL_SV/{vcf_name(ch)}" for b in G1K]
GEU_NAME = "GD462.GeneQuantRPKM.50FN.samplename.resk10.txt.gz"
# GEUVADIS processed files live in the "analysis_results" sub-folder of E-GEUV-1 (ArrayExpress -> BioStudies)
GEU_BASES = ["https://ftp.ebi.ac.uk/biostudies/fire/E-GEUV-/001/E-GEUV-1/Files",
             "https://ftp.ebi.ac.uk/biostudies/nfs/E-GEUV-/001/E-GEUV-1/Files",
             "https://ftp.ebi.ac.uk/biostudies/fire/E-GEUV-/E-GEUV-1/Files"]
GEU_URLS = ([f"{b}/analysis_results/{GEU_NAME}" for b in GEU_BASES] +
            [f"https://www.ebi.ac.uk/biostudies/files/E-GEUV-1/analysis_results/{GEU_NAME}",
             f"https://www.ebi.ac.uk/arrayexpress/files/E-GEUV-1/analysis_results/{GEU_NAME}",
             f"https://ftp.ebi.ac.uk/pub/databases/arrayexpress/data/experiment/GEUV/E-GEUV-1/analysis_results/{GEU_NAME}"] +
            [f"{b}/{GEU_NAME}" for b in GEU_BASES[:2]])
GEU_PATTERN = r"^GD46\d\.GeneQuantRPKM.*\.txt\.gz$"     # fallback: any GEUVADIS gene-level RPKM matrix
GTF_NAME = "gencode.v38.basic.annotation.gtf.gz"
GTF_URLS = [f"{b}/pub/databases/gencode/Gencode_human/release_38/{GTF_NAME}"
            for b in ["https://ftp.ebi.ac.uk", "http://ftp.ebi.ac.uk", "ftp://ftp.ebi.ac.uk"]]
CCRE_NAME = "GRCh38-cCREs.bed"
CCRE_URLS = ["https://downloads.wenglab.org/V3/GRCh38-cCREs.bed",
             "https://downloads.wenglab.org/cCREs/GRCh38-cCREs.bed"]

def biostudies_bases(acc):
    """Ask the BioStudies API where the study's files live (handles EBI path changes)."""
    out = []
    try:
        info = http_json(f"https://www.ebi.ac.uk/biostudies/api/v1/studies/{acc}/info")
        for k in ("ftpHttp_link", "ftpLink", "httpLink"):
            if info.get(k):
                base = info[k].rstrip("/")
                if base.startswith("ftp://"): base = "https://" + base[len("ftp://"):]
                out.append(base + ("" if base.endswith("/Files") else "/Files"))
    except Exception as e:
        log("[data] BioStudies API not reachable:", type(e).__name__)
    return out

def resolve_geuvadis_urls():
    bases = biostudies_bases("E-GEUV-1") + GEU_BASES
    bases = list(dict.fromkeys(bases))
    direct = [f"{b}/analysis_results/{GEU_NAME}" for b in bases]
    return direct + crawl_listing(bases, GEU_NAME, GEU_PATTERN, max_depth=2)


ASE_NAMES = ["ASE_MAGE.tar.xz", "MAGE_ASE.tar.xz"]
def ase_urls(n):
    return [f"https://github.com/PejLab/ASE_Data/raw/refs/heads/main/{n}",
            f"https://media.githubusercontent.com/media/PejLab/ASE_Data/main/{n}",
            f"https://raw.githubusercontent.com/PejLab/ASE_Data/main/{n}"]
def zenodo_ase_urls():
    out = []
    for rec in ("15226193",):
        try:
            js = http_json(f"https://zenodo.org/api/records/{rec}")
            for f in js.get("files", []):
                k = f.get("key", ""); L = f.get("links", {})
                if re.search(r"ase", k, re.I): out.append(L.get("self") or L.get("content"))
        except Exception as e:
            log("Zenodo ASE resolver:", type(e).__name__)
    return [u for u in out if u]

In [ ]:
# =====================================================================
# 0d. SELF-TEST synthetic data (same formats as the real files)
# =====================================================================
def make_synthetic_v2(seed=11):
    rng = np.random.default_rng(seed)
    pops = [("CEU", "EUR"), ("YRI", "AFR"), ("CHB", "EAS"), ("PEL", "AMR"), ("GIH", "SAS")]
    rows, samples = [], []
    sid = 100
    for p, sp in pops:
        for _ in range(110):
            s = f"NA{sid:05d}"; sid += 1; rows.append((s, s, "0", "0", rng.integers(1, 3), p, sp)); samples.append(s)
    geu_only = [f"HG{(90000+i):05d}" for i in range(150)]
    rows += [(s, s, "0", "0", 1, "GBR", "EUR") for s in geu_only]
    ped = pd.DataFrame(rows, columns=["FamilyID", "SampleID", "FatherID", "MotherID", "Sex", "Population", "Superpopulation"])
    vcf_s = samples + geu_only
    ped.to_csv(DATA / PED_NAME, sep=" ", index=False)
    n_genes = 80; tss = np.linspace(17_500_000, 49_000_000, n_genes).astype(int)
    pos = np.unique(np.concatenate([t + rng.integers(-CFG["CIS_WINDOW"], CFG["CIS_WINDOW"], 90) for t in tss]))
    nv = len(pos)
    # LD: mosaic of founders in blocks
    F = (rng.random((10, nv)) < rng.uniform(0.1, 0.6, nv)).astype(np.int8)
    H = np.zeros((len(vcf_s), nv, 2), np.int8)
    for i in range(len(vcf_s)):
        for c in range(2):
            for s0 in range(0, nv, 25): H[i, s0:s0 + 25, c] = F[rng.integers(10), s0:s0 + 25]
    H = np.where(rng.random(H.shape) < 0.03, 1 - H, H).astype(np.int8)
    with gzip.open(DATA / vcf_name("chr22"), "wt") as f:
        f.write("##fileformat=VCFv4.2\n#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\t" + "\t".join(vcf_s) + "\n")
        for v in range(nv):
            f.write(f"chr22\t{pos[v]}\t.\tA\tG\t.\tPASS\t.\tGT\t" + "\t".join(f"{x}|{y}" for x, y in H[:, v]) + "\n")
    genes = [f"ENSG{(200000+i):011d}" for i in range(n_genes)]
    cells = np.empty((n_genes, len(samples)), object); truth = []
    E = np.zeros((n_genes, len(geu_only))); Hg_ = H[len(samples):]; H = H[:len(samples)]
    for g in range(n_genes):
        w = np.where(np.abs(pos - tss[g]) < CFG["CIS_WINDOW"])[0]
        af = H[:, w].mean((0, 2)); w = w[(af > 0.2) & (af < 0.8)]
        j, k = rng.choice(w, 2, replace=False); b = rng.normal(0, 0.6, 2)
        gam = [0.0, 0.0, 1.0, -1.0][g % 4] if g % 2 == 0 else 0.0
        if g % 8 == 0: b = np.array([0.7, -0.7]); gam = 0.8          # planted compensation-like pairs
        lA = b[0] * H[:, j, 0] + b[1] * H[:, k, 0] + gam * H[:, j, 0] * H[:, k, 0]
        lB = b[0] * H[:, j, 1] + b[1] * H[:, k, 1] + gam * H[:, j, 1] * H[:, k, 1]
        p = 1 / (1 + np.exp(-(lA - lB + rng.normal(0, 0.3, len(samples)))))
        n = rng.poisson(rng.uniform(5, 80))
        n = rng.poisson(np.full(len(samples), max(n, 5)))
        a = rng.binomial(n, p)
        cells[g] = [f"{x},{y}" for x, y in zip(a, n - a)]
        truth.append((genes[g], pos[j], pos[k], gam, b[0], b[1]))
        eA = np.exp(b[0] * Hg_[:, j, 0] + b[1] * Hg_[:, k, 0] + gam * Hg_[:, j, 0] * Hg_[:, k, 0])
        eB = np.exp(b[0] * Hg_[:, j, 1] + b[1] * Hg_[:, k, 1] + gam * Hg_[:, j, 1] * Hg_[:, k, 1])
        E[g] = np.log(eA + eB) + rng.normal(0, 0.4, len(geu_only))
    os.makedirs(DATA / "MAGE_ASE", exist_ok=True)
    df = pd.DataFrame(cells, columns=samples); df.insert(0, "gene_id", [g + ".5" for g in genes])
    df.to_csv(DATA / "MAGE_ASE" / "MAGE_haplevel.csv", index=False)
    with tarfile.open(DATA / ASE_NAMES[0], "w:xz") as t: t.add(DATA / "MAGE_ASE", arcname="MAGE_ASE")
    with gzip.open(DATA / GTF_NAME, "wt") as f:
        for g, t in zip(genes, tss):
            f.write(f'chr22\tS\tgene\t{t}\t{t+20000}\t.\t+\t.\tgene_id "{g}.1"; gene_type "protein_coding"; gene_name "S{g[-3:]}";\n')
    ge = pd.DataFrame(E, columns=geu_only); ge.insert(0, "TargetID", [g + ".1" for g in genes]); ge.insert(1, "Gene_Symbol", genes)
    ge.insert(2, "Chr", "22"); ge.insert(3, "Coord", tss); ge.to_csv(DATA / GEU_NAME, sep="\t", index=False, compression="gzip")
    cc = np.sort(rng.choice(pos, 300, replace=False))
    pd.DataFrame({"c": "chr22", "s": cc - 150, "e": cc + 150}).to_csv(DATA / CCRE_NAME, sep="\t", header=False, index=False)
    pd.DataFrame(truth, columns=["gene", "pos1", "pos2", "gamma", "b1", "b2"]).to_csv(DATA / "synthetic_truth_v2.csv", index=False)
    tt_all = pd.DataFrame(truth, columns=["gene", "pos1", "pos2", "gamma", "b1", "b2"])
    sig_ = (tt_all.gamma != 0).values
    pd.DataFrame({"chrom": "22", "x_pos": tt_all.pos1, "y_pos": tt_all.pos2, "x_ref": "A", "x_alt": "G",
                  "eps_cell": np.where(sig_, tt_all.gamma * 0.7, 0) + rng.normal(0, 0.05, len(tt_all)), "se_cell": 0.08,
                  "p_cell": np.where(sig_, 1e-4, 0.5), "q_cell": np.where(sig_, 1e-3, 0.8), "n_meas_cell": 3,
                  "eps_all": np.where(sig_, tt_all.gamma * 0.6, 0) + rng.normal(0, 0.05, len(tt_all)), "se_all": 0.08,
                  "p_all": np.where(sig_, 1e-4, 0.6), "q_all": np.where(sig_, 1e-3, 0.9), "n_meas_all": 9}).to_csv(DATA / "mpra_pairs_meta.csv.gz", index=False)
    (DATA / "experimental").mkdir(exist_ok=True)
    tt = pd.DataFrame(truth, columns=["gene", "pos1", "pos2", "gamma", "b1", "b2"]); tt = tt[tt.gamma != 0].head(6)
    pd.DataFrame({"chrom": "chr22", "pos1": tt.pos1, "pos2": tt.pos2, "eps": tt.gamma, "eps_se": 0.1}).to_csv(
        DATA / "experimental" / "synthetic_mpra.csv", index=False)
    pd.DataFrame({"chrom": "chr22", "pos1": tt.pos1[:2], "pos2": tt.pos2[:2], "perturb_effect": tt.gamma[:2],
                  "perturb_p": 0.001}).to_csv(DATA / "experimental" / "synthetic_crispr.csv", index=False)
    log("SELF_TEST v2 synthetic data written")
if CFG["SELF_TEST"] and CFG["MODE"] == "BATCH":
    make_synthetic_v2()

## 2. Data: pedigree, MAGE haplotype-level ASE, annotation, cCREs, GEUVADIS (BATCH mode)

In [ ]:
def load_pedigree():
    p = fetch(PED_NAME, PED_URLS, min_bytes=100 if CFG["SELF_TEST"] else 10_000, desc="1000G pedigree")
    raw = pd.read_csv(p, sep=r"\s+", dtype=str)
    col = lambda *k, ex=(): next((c for c in raw.columns if all(x in c.lower() for x in k)
                                  and not any(e in c.lower() for e in ex)), None)
    P = pd.DataFrame({"sample": raw[col("sample")], "family": raw[col("fam")], "father": raw[col("father")],
                      "mother": raw[col("mother")], "pop": raw[col("pop", ex=("super",))], "superpop": raw[col("super")]})
    return P.drop_duplicates("sample").set_index("sample", drop=False)

def parse_pair_matrix(path):
    """MAGE haplotype-level ASE: rows = genes (Ensembl), columns = samples, each cell a pair of counts
       (e.g. '12,7' / '12|7' / '(12, 7)'). Returns gene ids, sample ids, A, B (int arrays)."""
    df = pd.read_csv(path, dtype=str, low_memory=False)
    gcol = next((c for c in df.columns if df[c].astype(str).str.match(r"^ENSG\d+").mean() > 0.5), None)
    if gcol is None:
        df = pd.read_csv(path, dtype=str, index_col=0, low_memory=False).reset_index(); gcol = df.columns[0]
    scols = [c for c in df.columns if re.match(r"^(HG|NA|GM)\d{5}", str(c))]
    if len(scols) < 20: raise ValueError(f"could not find sample columns in {path.name}: {list(df.columns[:8])}")
    genes = df[gcol].astype(str).str.replace(r"\.\d+$", "", regex=True).values
    A = np.zeros((len(df), len(scols)), np.int32); B = np.zeros_like(A)
    for j, c in enumerate(scols):
        ext = df[c].astype(str).str.extract(r"(\d+)\D+(\d+)")
        A[:, j] = pd.to_numeric(ext[0], errors="coerce").fillna(0).astype(np.int32).values
        B[:, j] = pd.to_numeric(ext[1], errors="coerce").fillna(0).astype(np.int32).values
    return genes, [str(c)[:7] for c in scols], A, B

def load_ase():
    p = None
    for n in ASE_NAMES:
        p = fetch(n, ase_urls(n), min_bytes=1000 if CFG["SELF_TEST"] else 100_000, required=False,
                  desc="MAGE haplotype-level ASE (PejLab)", resolvers=[zenodo_ase_urls])
        if p is not None: break
    if p is None:
        # an already-extracted CSV attached as Kaggle input also works
        hits = list(Path("/kaggle/input").rglob("*haplevel*.csv")) if Path("/kaggle/input").exists() else []
        if not hits: raise RuntimeError("MAGE ASE not obtained. Attach ASE_MAGE.tar.xz or MAGE_haplevel.csv "
                                        "(github.com/PejLab/ASE_Data) as a Kaggle Dataset.")
        csv = hits[0]
    else:
        ex = DATA / "ase_extracted"; ex.mkdir(exist_ok=True)
        if not any(ex.rglob("*.csv")):
            with tarfile.open(p) as t:
                for m in t.getmembers():
                    log(f"   archive member: {m.name} ({m.size/1e6:.1f} MB)")
                t.extractall(ex)
        cands = sorted(ex.rglob("*.csv"))
        csv = next((c for c in cands if "hap" in c.name.lower()), None)
        if csv is None: raise RuntimeError(f"no haplotype-level CSV in archive: {[c.name for c in cands]}")
    log("ASE file:", csv)
    genes, samples, A, B = parse_pair_matrix(csv)
    log(f"ASE matrix: {len(genes)} genes x {len(samples)} samples; observations with >= {CFG['MIN_COUNT']} reads: "
        f"{int(((A + B) >= CFG['MIN_COUNT']).sum()):,}")
    with open(csv) as f: head = [next(f)[:300] for _ in range(3)]
    return genes, samples, A, B, head

def parse_gtf_genes(path, chroms):
    rows = []; cb = set(chroms) | {c.replace("chr", "") for c in chroms}
    with gzip.open(path, "rt") as f:
        for line in f:
            if line.startswith("#"): continue
            t = line.split("\t", 9)
            if len(t) < 9 or t[2] != "gene" or t[0] not in cb: continue
            gid = re.search(r'gene_id "([^"]+)"', t[8]).group(1).split(".")[0]
            gn = re.search(r'gene_name "([^"]+)"', t[8])
            st, en, sd = int(t[3]), int(t[4]), t[6]
            rows.append(dict(gene_id=gid, gene_name=gn.group(1) if gn else gid,
                             chrom=t[0] if t[0].startswith("chr") else "chr" + t[0], strand=sd, tss=st if sd == "+" else en))
    return pd.DataFrame(rows).drop_duplicates("gene_id").set_index("gene_id", drop=False)

def load_ccre(chroms):
    out = {}
    p = fetch(CCRE_NAME, CCRE_URLS, min_bytes=100 if CFG["SELF_TEST"] else 1_000_000, required=False, desc="ENCODE cCREs")
    if p is None: return out
    cc = pd.read_csv(p, sep="\t", header=None, usecols=[0, 1, 2], names=["chrom", "start", "end"])
    for ch in chroms:
        s = cc[cc.chrom == ch].sort_values("start"); out[ch] = (s.start.values, s.end.values)
    return out
def in_ccre(CC, chrom, pos):
    if chrom not in CC or len(CC[chrom][0]) == 0: return np.zeros(len(pos), bool)
    s, e = CC[chrom]; i = np.searchsorted(s, pos, side="right") - 1; ok = i >= 0
    o = np.zeros(len(pos), bool); o[ok] = pos[ok] <= e[i[ok]]; return o

if CFG["MODE"] == "BATCH":
    PED = load_pedigree()
    ASE_GENES, ASE_SAMPLES, ASE_A, ASE_B, ASE_HEAD = load_ase()
    print("First lines of the ASE file (format check):"); [print("  ", h.strip()[:200]) for h in ASE_HEAD]
    gtf = fetch(GTF_NAME, GTF_URLS, min_bytes=100 if CFG["SELF_TEST"] else 1_000_000, desc="GENCODE v38")
    ANNOT = parse_gtf_genes(gtf, CHROMS)
    CC = load_ccre(CHROMS) if CFG["USE_CCRE"] else {}
    GEU = None
    if CFG["USE_GEUVADIS"]:
        gp = fetch(GEU_NAME, GEU_URLS, min_bytes=100 if CFG["SELF_TEST"] else 100_000, required=False,
                   desc="GEUVADIS expression", resolvers=[resolve_geuvadis_urls])
        if gp is not None:
            g = pd.read_csv(gp, sep="\t")
            sc = [c for c in g.columns if re.match(r"^(HG|NA|GM)\d{5}$", str(c))]
            GEU = g[sc].apply(pd.to_numeric, errors="coerce"); GEU.index = g.iloc[:, 0].astype(str).str.replace(r"\.\d+$", "", regex=True)
    gi = {g: i for i, g in enumerate(ASE_GENES)}
    GENES = ANNOT[ANNOT.index.isin(gi)].sort_values(["chrom", "tss"]).copy()
    log(f"Genes with ASE on {CHROMS}: {len(GENES)}")

## 3. Phased genotypes for the batch's chromosomes (streamed; each VCF is deleted after parsing to save disk)

In [ ]:
def merged_windows(tss, w):
    iv = sorted((max(1, t - w), t + w) for t in tss)
    out = []
    for s, e in iv:
        if out and s <= out[-1][1]: out[-1][1] = max(out[-1][1], e)
        else: out.append([s, e])
    return np.array([x[0] for x in out]), np.array([x[1] for x in out])

def stream_lines(path):
    if shutil.which("gzip"):
        p = subprocess.Popen(["gzip", "-dc", str(path)], stdout=subprocess.PIPE, bufsize=1 << 22)
        try:
            for line in p.stdout: yield line
        finally:
            p.stdout.close(); p.kill()
    else:
        with gzip.open(path, "rb") as f:
            for line in f: yield line

def parse_vcf_windows(path, chrom, starts, ends, keep, maf_samples, maf_min):
    keep = list(keep)
    names, sel, maf_idx = None, None, None
    POS, ID, REF, ALT, H0, H1 = [], [], [], [], [], []
    n_lines = n_win = n_snv = n_slow = 0
    cb = {chrom.encode(), chrom.replace("chr", "").encode()}
    last_end = ends[-1]; t0 = time.time()
    for line in stream_lines(path):
        if line[:2] == b"##": continue
        if line[:1] == b"#":
            names = [x.decode() for x in line.rstrip(b"\r\n").split(b"\t")[9:]]
            nidx = {s: i for i, s in enumerate(names)}
            keep = [s for s in keep if s in nidx]
            sel = np.array([nidx[s] for s in keep], int)
            kidx = {s: i for i, s in enumerate(keep)}
            maf_idx = np.array([kidx[s] for s in maf_samples if s in kidx], int)
            nS = len(names)
            log(f"  VCF {chrom}: {nS} samples in file, keeping {len(keep)}; MAF computed on {len(maf_idx)}")
            continue
        n_lines += 1
        if n_lines % 250_000 == 0:
            print(f"   ... {n_lines:,} records, {n_win:,} in windows, {n_snv:,} kept ({time.time()-t0:.0f}s)", flush=True)
        t = line.split(b"\t", 9)
        if t[0] not in cb: continue
        pos = int(t[1])
        if pos > last_end: break
        i = bisect.bisect_right(starts, pos) - 1
        if i < 0 or pos > ends[i]: continue
        n_win += 1
        ref, alt = t[3], t[4]
        if len(ref) != 1 or len(alt) != 1 or alt not in (b"A", b"C", b"G", b"T") or ref not in (b"A", b"C", b"G", b"T"):
            continue
        blob = t[9].rstrip(b"\r\n")
        if t[8] == b"GT" and len(blob) == 4 * nS - 1:
            arr = np.frombuffer(blob, dtype=np.uint8)
            sep = arr[sel * 4 + 1]
            a0 = arr[sel * 4].astype(np.int16) - 48; a1 = arr[sel * 4 + 2].astype(np.int16) - 48
            if (sep != 124).any() or (a0 < 0).any() or (a0 > 1).any() or (a1 < 0).any() or (a1 > 1).any(): continue
        else:
            n_slow += 1
            f = blob.split(b"\t")
            g = [f[j].split(b":")[0] for j in sel]
            if any(len(x) != 3 or x[1:2] != b"|" or x[0:1] not in (b"0", b"1") or x[2:3] not in (b"0", b"1") for x in g):
                continue
            a0 = np.array([x[0] - 48 for x in g], np.int16); a1 = np.array([x[2] - 48 for x in g], np.int16)
        af = (a0[maf_idx].sum() + a1[maf_idx].sum()) / (2 * len(maf_idx))
        if min(af, 1 - af) < maf_min: continue
        n_snv += 1
        POS.append(pos); ID.append(t[2].decode()); REF.append(ref.decode()); ALT.append(alt.decode())
        H0.append(a0.astype(np.int8)); H1.append(a1.astype(np.int8))
    log(f"  VCF {chrom}: {n_lines:,} records scanned, {n_win:,} in windows, {n_snv:,} biallelic phased SNVs kept "
        f"(slow-path {n_slow}) in {time.time()-t0:.0f}s")
    H = np.stack([np.stack(H0), np.stack(H1)], -1) if POS else np.zeros((0, len(keep), 2), np.int8)
    var = pd.DataFrame(dict(chrom=chrom, pos=np.array(POS, int), rsid=ID, ref=REF, alt=ALT))
    return var, H, keep


if CFG["MODE"] == "BATCH":
    in_ped = set(PED.index)
    ase_s = [s for s in ASE_SAMPLES if s in in_ped or CFG["SELF_TEST"]]
    geu_only = [s for s in (GEU.columns if GEU is not None else []) if s not in set(ASE_SAMPLES)]
    keep = sorted(set(ase_s) | set(geu_only))
    VAR_L, HAP_L, VCF_SAMPLES = [], [], None
    for ch in CHROMS:
        gch = GENES[GENES.chrom == ch]
        if len(gch) == 0: continue
        vp = fetch(vcf_name(ch), vcf_urls(ch), min_bytes=100 if CFG["SELF_TEST"] else 20_000_000, desc=f"1000G 30x {ch}",
                   resolvers=[lambda ch=ch: crawl_listing([f"{b}/{HC}/working" for b in G1K[:3]], vcf_name(ch), None, 1)])
        s, e = merged_windows(gch.tss.values, CFG["CIS_WINDOW"])
        v, H, kept = parse_vcf_windows(vp, ch, s, e, keep, ase_s, CFG["MAF_MIN"])
        if VCF_SAMPLES is None: VCF_SAMPLES = kept
        VAR_L.append(v); HAP_L.append(H)
        if not CFG["SELF_TEST"] and IS_KAGGLE:
            try: vp.unlink(); log(f"  deleted {vp.name} to free disk")
            except Exception: pass
        gc.collect()
    VAR = pd.concat(VAR_L, ignore_index=True); HAP = np.concatenate(HAP_L, 0); del HAP_L
    VAR["vid"] = VAR.chrom + ":" + VAR.pos.astype(str) + ":" + VAR.ref + ":" + VAR.alt
    VAR["ccre"] = False
    for ch in VAR.chrom.unique():
        m = (VAR.chrom == ch).values; VAR.loc[m, "ccre"] = in_ccre(CC, ch, VAR.pos.values[m])
    VSIDX = {s: i for i, s in enumerate(VCF_SAMPLES)}
    log(f"Variants: {len(VAR):,} | HAP {HAP.shape} ({HAP.nbytes/1e9:.2f} GB)")

## 4. Cohort, families, splits, and the phase-alignment check (ASE haplotype A ↔ VCF haplotype 0)

In [ ]:
def make_folds(groups, strata, k, seed=0):
    rng = np.random.default_rng(seed)
    fams = pd.DataFrame({"g": groups, "s": strata}).groupby("g").agg(s=("s", "first"), n=("s", "size")).reset_index()
    fold_of, load = {}, np.zeros(k)
    for _, sub in fams.groupby("s"):
        sub = sub.sample(frac=1, random_state=int(rng.integers(1e9))).sort_values("n", ascending=False, kind="stable")
        loc = np.zeros(k)
        for _, r in sub.iterrows():
            j = int(np.lexsort((load, loc))[0]); fold_of[r.g] = j; loc[j] += r.n; load[j] += r.n
    return np.array([fold_of[g] for g in groups])

def emp_logit(a, b): return np.log((a + 0.5) / (b + 0.5))
def emp_w(a, b): return 1.0 / (1.0 / (a + 0.5) + 1.0 / (b + 0.5))

def wcorr_cols(X, y, w):
    """weighted correlation of y with each column of X (vectorised)."""
    w = w / w.sum(); mx = w @ X; my = w @ y
    Xc = X - mx; yc = y - my
    cov = (w[:, None] * Xc * yc[:, None]).sum(0)
    vx = (w[:, None] * Xc ** 2).sum(0); vy = (w * yc ** 2).sum()
    return cov / np.sqrt(vx * vy + 1e-300)

if CFG["MODE"] == "BATCH":
    S = [s for s in ASE_SAMPLES if s in VSIDX]
    S_V = np.array([VSIDX[s] for s in S]); colS = np.array([ASE_SAMPLES.index(s) for s in S])
    nS = len(S)
    def fam_of(s):
        r = PED.loc[s] if s in PED.index else None
        return str(r.family) if r is not None and isinstance(r.family, str) else s
    FAM = np.array([fam_of(s) for s in S])
    SUPER = np.array([PED.loc[s, "superpop"] if s in PED.index else "UNK" for s in S])
    POP = np.array([PED.loc[s, "pop"] if s in PED.index else "UNK" for s in S])
    HALF = make_folds(FAM, SUPER, 2, seed=99)
    FOLDS = make_folds(FAM, SUPER, CFG["N_FOLDS"], seed=2026)
    W = CFG["CIS_WINDOW"]
    gene_rows, GVARS = [], []
    gidx = {g: i for i, g in enumerate(ASE_GENES)}
    for gid, r in GENES.iterrows():
        m = np.where((VAR.chrom.values == r.chrom) & (np.abs(VAR.pos.values - r.tss) <= W))[0]
        n_obs = ((ASE_A[gidx[gid], colS] + ASE_B[gidx[gid], colS]) >= CFG["MIN_COUNT"])
        if len(m) >= 3 and n_obs[HALF == 0].sum() >= CFG["MIN_IND"]:
            gene_rows.append(gid); GVARS.append(m)
    GENES = GENES.loc[gene_rows].copy()
    if CFG["MAX_GENES"]: GENES = GENES.iloc[:CFG["MAX_GENES"]]; GVARS = GVARS[:CFG["MAX_GENES"]]
    G = len(GENES)
    GA = np.stack([ASE_A[gidx[g], colS] for g in GENES.index]); GB = np.stack([ASE_B[gidx[g], colS] for g in GENES.index])
    log(f"ASE cohort: {nS} individuals ({dict(zip(*np.unique(SUPER, return_counts=True)))}), "
        f"{len(set(FAM))} families | genes tested: {G} | discovery/replication: {(HALF==0).sum()}/{(HALF==1).sum()}")

    # ---- phase-alignment check ----
    # If the ASE haplotype counts are oriented like the VCF haplotypes, many genes show a strong SIGNED association
    # between the allelic log-ratio and (hA - hB) of some cis variant. With random orientation that association vanishes
    # (a global A<->B swap only flips all signs and changes no test). The check also reports the phase-free association.
    rng_al = np.random.default_rng(0); al_rows = []
    for g in rng_al.choice(G, min(G, 400), replace=False):
        H = HAP[GVARS[g]][:, S_V, :]; dA = (H[:, :, 0].astype(float) - H[:, :, 1]).T
        a, b = GA[g], GB[g]; ok = (a + b) >= CFG["MIN_COUNT"]
        if ok.sum() < 30: continue
        L = emp_logit(a[ok], b[ok]); w = emp_w(a[ok], b[ok])
        r_s = wcorr_cols(dA[ok], L, w); r_u = wcorr_cols(np.abs(dA[ok]), np.abs(L), w)
        n = ok.sum()
        al_rows.append(dict(gene=GENES.index[g], n=n, max_signed_r=np.nanmax(np.abs(r_s)), max_phasefree_r=np.nanmax(np.abs(r_u)),
                            strong_signed=np.nanmax(np.abs(r_s)) * np.sqrt(n) > 5.0))
    ALIGN = pd.DataFrame(al_rows); save_table(ALIGN, f"S_align_batch{CFG['BATCH']}")
    frac_strong = float(ALIGN.strong_signed.mean()) if len(ALIGN) else 0.0
    log(f"Phase-alignment check: {frac_strong:.1%} of genes with |z| > 5 for a signed haplotype-ASE association "
        f"(median max |r| signed={ALIGN.max_signed_r.median():.3f}, phase-free={ALIGN.max_phasefree_r.median():.3f})")
    if frac_strong < CFG["ALIGN_MIN_FRAC"]:
        raise SystemExit("ASE haplotype counts do not appear aligned with the 1000G 30x VCF haplotypes "
                         "(no signed cis association). A cis-epistasis analysis would be invalid. "
                         "Send the 'First lines of the ASE file' printout above so the parser can be adapted.")

## 5. Discovery → replication cis-epistasis scan (quasi-binomial GLM), negative controls, GEUVADIS cross-assay

In [ ]:
def independent_columns(X, priority, tol=1e-8):
    """Greedy rank check: add columns in `priority` order and keep a column only if it raises the rank.
       Collinear columns (perfect LD, cis term identical to a main effect, ...) are dropped instead of being
       fitted with a pseudo-inverse, which would give non-unique, unreliable estimates."""
    keep = []
    for j in priority:
        if j > 0 and X[:, j].std() <= 1e-9: continue
        cand = keep + [j]
        if np.linalg.matrix_rank(X[:, cand], tol=tol * max(1.0, np.abs(X[:, cand]).max())) == len(cand): keep = cand
    return keep

def qb_fit(a, b, X, priority=None):
    """Binomial GLM (logit) with ROBUST sandwich (HC0) standard errors.
       Self-test calibration: quasi-binomial SEs gave lambda_GC ~33 under the permutation null (genotype-dependent
       overdispersion); sandwich SEs gave lambda ~1.0. Linearly dependent columns are removed first (priority order:
       intercept, tested terms, then conditioning variants). Returns dict col->(est, se, p)."""
    priority = priority if priority is not None else list(range(X.shape[1]))
    keep = independent_columns(X, priority)
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            m = sm.GLM(np.column_stack([a, b]), X[:, keep], family=sm.families.Binomial()).fit(cov_type="HC0", maxiter=50)
        est, se = np.asarray(m.params), np.asarray(m.bse)
        if not np.all(np.isfinite(se)): return {}
        z = est / se
        return {c: (est[i], se[i], 2 * norm.sf(abs(z[i]))) for i, c in enumerate(keep)}
    except Exception:
        return {}

def clump(Z, r, ok, K, thr):
    order = np.argsort(-np.abs(r)); ch = []; n = Z.shape[0]
    for j in order:
        if not ok[j]: continue
        if ch and np.max(((Z[:, ch].T @ Z[:, j]) / n) ** 2) >= thr: continue
        ch.append(j)
        if len(ch) >= K: break
    return np.array(ch, int)

def pair_test(hA, hB, a, b, j, k, cond):
    """returns (beta_j, beta_k from the additive model), gamma (cis-epistasis), informative n."""
    dj, dk = hA[:, j] - hB[:, j], hA[:, k] - hB[:, k]
    cis = hA[:, j] * hA[:, k] - hB[:, j] * hB[:, k]
    n_inf = int((cis != 0).sum())
    if n_inf < CFG["MIN_INFORMATIVE"]: return None
    C = np.column_stack([hA[:, c] - hB[:, c] for c in cond if c not in (j, k)][:CFG["N_COND"]]) \
        if any(c not in (j, k) for c in cond) else np.zeros((len(a), 0))
    one = np.ones((len(a), 1))
    add = qb_fit(a, b, np.column_stack([one, dj, dk, C]))
    Xf = np.column_stack([one, dj, dk, cis, C])
    full = qb_fit(a, b, Xf, priority=[0, 1, 2, 3] + list(range(4, Xf.shape[1])))
    if 3 not in full or 1 not in full or 2 not in full: return None     # gamma not identifiable separately from main effects
    return dict(beta1=add.get(1, (np.nan,))[0], beta2=add.get(2, (np.nan,))[0], gamma=full[3][0], gamma_se=full[3][1],
                p=full[3][2], n_inf=n_inf, n=len(a))

def ase_gene(g, Hg, a_all, b_all, half, vids, mafs, seed):
    """Hg: n x p x 2 (all ASE individuals). Discovery selection, pair tests in both halves, controls."""
    rng = np.random.default_rng(seed + g)
    out = []
    ok_all = (a_all + b_all) >= CFG["MIN_COUNT"]
    disc = ok_all & (half == 0); rep = ok_all & (half == 1)
    hA = Hg[:, :, 0].astype(float); hB = Hg[:, :, 1].astype(float); dA = hA - hB
    a, b = a_all[disc], b_all[disc]
    L = emp_logit(a, b); w = emp_w(a, b)
    D = (hA + hB)[disc]; sd = D.std(0); okv = sd > 1e-9
    Zs = np.zeros_like(D); Zs[:, okv] = (D[:, okv] - D[:, okv].mean(0)) / sd[okv]
    r = np.nan_to_num(wcorr_cols(dA[disc], L, w)); r[~okv] = 0
    cand = clump(Zs, r, okv, CFG["K_PAIR"], CFG["PAIR_MAX_R2"])
    cond = clump(Zs, r, okv, CFG["N_COND"] + 2, CFG["CLUMP_R2"])
    def run(pairs, kind, flip=None):
        for j, k in pairs:
            res = {}
            for lab, m in (("disc", disc), ("rep", rep)):
                if kind != "real" and lab == "rep": continue
                aa, bb = a_all[m], b_all[m]
                if flip is not None and lab == "disc":
                    f = flip[m]; aa, bb = np.where(f, bb, aa), np.where(f, aa, bb)
                t = pair_test(hA[m], hB[m], aa, bb, j, k, cond)
                if t is None: break
                res[lab] = t
            if "disc" not in res: continue
            row = dict(gene=GENES.index[g], kind=kind, v1=vids[j], v2=vids[k],
                       r2_ld=float((Zs[:, j] @ Zs[:, k] / len(Zs)) ** 2), **{f"disc_{x}": y for x, y in res["disc"].items()})
            if "rep" in res: row.update({f"rep_{x}": y for x, y in res["rep"].items()})
            out.append(row)
    pairs = [(j, k) for j, k in itertools.combinations(cand, 2)]
    run(pairs, "real")
    for pi in range(CFG["N_PERM"]):
        run(pairs, f"perm{pi}", flip=rng.random(len(a_all)) < 0.5)
    pool = np.setdiff1d(np.where(okv)[0], cand)
    if len(pool) >= 4:
        rp = []
        for _ in range(CFG["N_RANDOM_PAIRS"]):
            j, k = rng.choice(pool, 2, replace=False)
            if (Zs[:, j] @ Zs[:, k] / len(Zs)) ** 2 < CFG["PAIR_MAX_R2"]: rp.append((j, k))
        run(rp, "random")
    return out

if CFG["MODE"] == "BATCH":
    t_scan = time.time()
    VID = VAR.vid.values; MAFV = None
    rows = Parallel(n_jobs=N_JOBS, backend="loky", batch_size=8, verbose=2)(
        delayed(ase_gene)(g, np.ascontiguousarray(HAP[GVARS[g]][:, S_V, :].transpose(1, 0, 2)), GA[g], GB[g], HALF,
                          VID[GVARS[g]], None, 7) for g in range(G))
    SCAN = pd.DataFrame([r for rr in rows for r in rr])
    log(f"Scan finished in {(time.time()-t_scan)/60:.1f} min: {len(SCAN):,} tests "
        f"({SCAN.kind.value_counts().to_dict() if len(SCAN) else {}})")
    # GEUVADIS cross-assay replication (individuals NOT in MAGE; total expression; cis-pair coefficient)
    GEU_REP = []
    if GEU is not None and len(SCAN):
        gs = [s for s in GEU.columns if s in VSIDX and s not in set(S)]
        top = SCAN[(SCAN.kind == "real") & (SCAN.disc_p < CFG["GEU_TEST_P"])]
        log(f"GEUVADIS-only individuals with genotypes: {len(gs)}; pairs to test: {len(top)}")
        if len(gs) >= 40 and len(top):
            gv = np.array([VSIDX[s] for s in gs]); vpos = {v: i for i, v in enumerate(VID)}
            for _, rr in top.iterrows():
                if rr.gene not in GEU.index: continue
                y = GEU.loc[rr.gene, gs].values.astype(float)
                if np.isnan(y).all() or np.nanstd(y) == 0: continue
                y = norm.ppf((rankdata(np.nan_to_num(y, nan=np.nanmedian(y))) - 0.5) / len(y))
                h1, h2 = HAP[vpos[rr.v1]][gv].astype(float), HAP[vpos[rr.v2]][gv].astype(float)
                d1, d2 = h1.sum(1), h2.sum(1)
                cis = h1[:, 0] * h2[:, 0] + h1[:, 1] * h2[:, 1]; trn = h1[:, 0] * h2[:, 1] + h1[:, 1] * h2[:, 0]
                X = sm.add_constant(np.column_stack([d1, d2, h1[:, 0] * h1[:, 1], h2[:, 0] * h2[:, 1], cis, trn]))
                try:
                    m = sm.OLS(y, X).fit(cov_type="HC3"); bp = np.asarray(m.params); V = np.asarray(m.cov_params())
                    if len(bp) < 7: continue
                    est, se = bp[5] - bp[6], np.sqrt(max(V[5, 5] + V[6, 6] - 2 * V[5, 6], 1e-12))
                except Exception:
                    continue
                z = est / se
                GEU_REP.append(dict(gene=rr.gene, v1=rr.v1, v2=rr.v2, geu_n=len(gs), geu_cis_minus_trans=est, geu_se=se,
                                    geu_p1=norm.sf(abs(z)) if np.sign(est) == np.sign(rr.disc_gamma) else 1 - norm.sf(abs(z))))
    GEU_REP = pd.DataFrame(GEU_REP)

## 6. Held-out prediction of allelic log-ratios (family-grouped CV): additive vs additive + sparse cis-pairs

In [ ]:
def wfit_ridge(X, y, w, Xte, alpha=1.0):
    sw = np.sqrt(w / w.mean()); mx = np.average(X, 0, w); my = np.average(y, weights=w)
    m = Ridge(alpha=alpha, fit_intercept=False).fit((X - mx) * sw[:, None], (y - my) * sw)
    return m.predict(Xte - mx) + my, (m.predict(X - mx) + my)

def wlasso_1se(X, y, w, groups, Xte):
    """weighted Lasso with family-grouped CV and the pre-registered 1-SE rule; returns test prediction, coef."""
    if X.shape[1] == 0 or len(set(groups)) < 5: return np.zeros(len(Xte)), np.zeros(X.shape[1])
    sw = np.sqrt(w / w.mean()); mx = np.average(X, 0, w); sd = np.sqrt(np.average((X - mx) ** 2, 0, w)); sd[sd < 1e-9] = 1
    A = (X - mx) / sd * sw[:, None]; t = y * sw
    amax = np.max(np.abs(A.T @ t)) / len(t) + 1e-12
    alphas = amax * np.logspace(0, -3, 25)
    errs = np.zeros((len(alphas), 5))
    for f, (tr, va) in enumerate(GroupKFold(5).split(A, t, groups)):
        for i, al in enumerate(alphas):
            m = Lasso(alpha=al, fit_intercept=False, max_iter=3000, tol=1e-3).fit(A[tr], t[tr])
            errs[i, f] = np.mean((t[va] - A[va] @ m.coef_) ** 2)
    mu, se = errs.mean(1), errs.std(1) / np.sqrt(5)
    i_min = int(np.argmin(mu)); i_1se = int(np.where(mu <= mu[i_min] + se[i_min])[0].min())   # largest alpha within 1 SE
    m = Lasso(alpha=alphas[i_1se], fit_intercept=False, max_iter=5000, tol=1e-4).fit(A, t)
    coef = m.coef_ / sd
    return (Xte - mx) @ coef, coef

def cis_feats(hA, hB, idx):
    P = list(itertools.combinations(range(len(idx)), 2))
    if not P: return np.zeros((len(hA), 0))
    i, j = np.array(P).T; a, b = hA[:, idx], hB[:, idx]
    return a[:, i] * a[:, j] - b[:, i] * b[:, j]

def pred_gene(g, Hg, a_all, b_all, folds, fam, seed):
    rng = np.random.default_rng(seed + g)
    ok = (a_all + b_all) >= CFG["MIN_COUNT"]
    hA = Hg[:, :, 0].astype(float)[ok]; hB = Hg[:, :, 1].astype(float)[ok]
    a, b = a_all[ok], b_all[ok]; L = emp_logit(a, b); w = emp_w(a, b); fo = folds[ok]; fm = fam[ok]
    sw_ = (rng.random(hA.shape) < 0.5) & (hA != hB)                    # phase-shuffled copy (control)
    hAs, hBs = np.where(sw_, hB, hA), np.where(sw_, hA, hB)
    P = {k: np.full(len(L), np.nan) for k in ("additive", "additive+cis", "additive+cis(shuffled phase)")}
    nsel = 0
    for k in range(CFG["N_FOLDS"]):
        tr, te = fo != k, fo == k
        if te.sum() < 5 or tr.sum() < 30: continue
        dA = hA - hB; D = (hA + hB)[tr]; sd = D.std(0); okv = sd > 1e-9
        Zs = np.zeros_like(D); Zs[:, okv] = (D[:, okv] - D[:, okv].mean(0)) / sd[okv]
        r = np.nan_to_num(wcorr_cols(dA[tr], L[tr], w[tr])); r[~okv] = 0
        sel = clump(Zs, r, okv, CFG["K_ADD"], CFG["CLUMP_R2"])
        if len(sel) == 0: continue
        # additive haplotype model, ridge penalty chosen on an inner family split
        best, best_a = np.inf, 1.0
        itr = np.isin(fm[tr], np.unique(fm[tr])[::5], invert=True)
        for al in (0.1, 1, 10, 100):
            p, _ = wfit_ridge(dA[tr][itr][:, sel], L[tr][itr], w[tr][itr], dA[tr][~itr][:, sel], al)
            e = np.average((L[tr][~itr] - p) ** 2, weights=w[tr][~itr]) if (~itr).sum() else 0
            if e < best: best, best_a = e, al
        p_te, p_tr = wfit_ridge(dA[tr][:, sel], L[tr], w[tr], dA[te][:, sel], best_a)
        P["additive"][te] = p_te
        kp = sel[:CFG["K_PAIR"]]
        for key, (xa, xb) in {"additive+cis": (hA, hB), "additive+cis(shuffled phase)": (hAs, hBs)}.items():
            F = cis_feats(xa, xb, kp)
            q, coef = wlasso_1se(F[tr], L[tr] - p_tr, w[tr], fm[tr], F[te])
            P[key][te] = p_te + q
            if key == "additive+cis": nsel += int((coef != 0).sum())
    res = dict(gene=GENES.index[g], n=len(L), n_cis_terms_selected=nsel)
    for k, p in P.items():
        m = ~np.isnan(p)
        if m.sum() < 20: res[f"R2_{k}"] = np.nan; continue
        yy, pp, ww = L[m], p[m], w[m]
        res[f"R2_{k}"] = 1 - np.sum(ww * (yy - pp) ** 2) / np.sum(ww * (yy - np.average(yy, weights=ww)) ** 2)
    return res

if CFG["MODE"] == "BATCH":
    t_p = time.time()
    PRED = pd.DataFrame(Parallel(n_jobs=N_JOBS, backend="loky", batch_size=8, verbose=2)(
        delayed(pred_gene)(g, np.ascontiguousarray(HAP[GVARS[g]][:, S_V, :].transpose(1, 0, 2)), GA[g], GB[g], FOLDS, FAM, 3)
        for g in range(G)))
    PRED["chrom"] = GENES.loc[PRED.gene, "chrom"].values; PRED["tss"] = GENES.loc[PRED.gene, "tss"].values
    log(f"Prediction CV finished in {(time.time()-t_p)/60:.1f} min")

    # ---- save batch outputs (inputs for COMBINE) ----
    tag = f"batch{CFG['BATCH']}" if not CFG["CHROMS_OVERRIDE"] else "pilot_" + "_".join(CHROMS)
    vmeta = VAR.set_index("vid")[["chrom", "pos", "ccre"]]
    for c in ("v1", "v2"):
        if len(SCAN):
            SCAN[f"{c}_ccre"] = vmeta.reindex(SCAN[c]).ccre.values
    if len(SCAN): SCAN["gene_name"] = GENES.loc[SCAN.gene, "gene_name"].values; SCAN["chrom"] = GENES.loc[SCAN.gene, "chrom"].values
    SCAN.to_csv(OUT / f"ase_scan_{tag}.csv.gz", index=False)
    PRED.to_csv(OUT / f"ase_pred_{tag}.csv.gz", index=False)
    GEU_REP.to_csv(OUT / f"geu_rep_{tag}.csv.gz", index=False)
    META = dict(tag=tag, chroms=CHROMS, n_individuals=nS, n_genes=G, n_variants=len(VAR), align_frac_strong=frac_strong,
                config_hash=cfg_hash(), runtime_min=round((time.time() - T0) / 60, 1), self_test=CFG["SELF_TEST"],
                superpops=dict(zip(*map(list, np.unique(SUPER, return_counts=True)))))
    with open(OUT / f"meta_{tag}.json", "w") as f: json.dump(META, f, indent=1, default=str)
    log("Batch outputs:", sorted(p.name for p in OUT.glob(f"*{tag}*")))

## 6b. Gap A — targeted test of MPRA-measured variant pairs (Siraj et al., Nature 2026) on natural haplotypes

The Siraj et al. MPRA libraries contain 2×2 constructs: focal variant X (ref/alt) × a neighbouring variant Y (ref/alt), in 3 oligo windows, with 5 replicates per cell type. For each pair this cell computes $\varepsilon = y_{11}-y_{10}-y_{01}+y_{00}$ (log2 RNA/DNA), meta-analyses it with random effects, auto-detects the genome build (lifting hg19 to GRCh38 if needed), and then tests exactly these pairs for $\gamma$ in MAGE haplotype ASE. Because the test is pre-specified and targeted, the multiple-testing burden is far smaller than in the genome-wide scan.

In [ ]:
SIRAJ_ZIP = "siraj2026_data_MPRA.zip"
SIRAJ_URLS = ["https://zenodo.org/api/records/15297965/files/data_MPRA.zip/content",
              "https://zenodo.org/records/15297965/files/data_MPRA.zip?download=1"]
MPRA_PAT = re.compile(r"^([0-9XY]+):(\d+):([ACGT]+):([ACGT]+):([AR]):(w[CLR])(?::Alt_([0-9XY]+)-(\d+)-([AR]))?$")

def mpra_eps_table(f):
    """2x2 interaction contrasts from one Siraj et al. normalized-count file (all cell types, all windows)."""
    d = pd.read_csv(f, sep="\t", index_col=0, low_memory=False)
    pl = [c for c in d.columns if c.lower().startswith("plasmid")]
    cts = sorted({re.sub(r"_r\d+$", "", c) for c in d.columns if c not in pl})
    key = {}
    for i, raw in enumerate(d.index.astype(str)):
        for sid in raw.strip("()").split(";"):
            m = MPRA_PAT.match(sid)
            if not m: continue
            c, p, r, a, xa, w, yc, yp, ya = m.groups()
            x = f"{c}:{p}:{r}:{a}"
            if yc is None: key.setdefault((x, xa, w, None, None), i)
            else: key[(x, xa, w, f"{yc}:{yp}", ya)] = i
    dna_mean = d[pl].mean(axis=1).values; dna = np.log2(dna_mean + 1)
    out = []
    for x, w, y in {(k[0], k[2], k[3]) for k in key if k[3] is not None}:
        ks = {(xa, ya): key.get((x, xa, w, y, ya)) for xa in "RA" for ya in "RA"}
        for xa in "RA":
            if ks[(xa, "R")] is None: ks[(xa, "R")] = key.get((x, xa, w, None, None))
        if any(v is None for v in ks.values()): continue
        dmin = min(dna_mean[v] for v in ks.values())
        if dmin < CFG["MPRA_MIN_DNA"]: continue
        for ct in cts:
            reps = [c for c in d.columns if re.sub(r"_r\d+$", "", c) == ct]
            act = {k: np.log2(d.iloc[v][reps].values.astype(float) + 1) - dna[v] for k, v in ks.items()}
            e = act[("A", "A")] - act[("A", "R")] - act[("R", "A")] + act[("R", "R")]; e = e[np.isfinite(e)]
            if len(e) < 2: continue
            se = e.std(ddof=1) / np.sqrt(len(e))
            if not se > 0: continue
            xc, xp, xr, xal = x.split(":"); yc, yp = y.split(":")
            out.append(dict(chrom=xc, x_pos=int(xp), x_ref=xr, x_alt=xal, y_pos=int(yp), cell_type=ct.lower(), window=w,
                            eps=e.mean(), eps_se=se))
    return pd.DataFrame(out)

def re_meta(e, se):
    """DerSimonian-Laird random-effects meta-analysis."""
    e, se = np.asarray(e, float), np.asarray(se, float); w = 1 / se ** 2
    m = (w * e).sum() / w.sum(); Q = (w * (e - m) ** 2).sum(); k = len(e)
    C = w.sum() - (w ** 2).sum() / w.sum(); tau2 = max(0.0, (Q - (k - 1)) / C) if C > 0 and k > 1 else 0.0
    ws = 1 / (se ** 2 + tau2); mm = (ws * e).sum() / ws.sum(); ss = np.sqrt(1 / ws.sum())
    return mm, ss, 2 * norm.sf(abs(mm / ss)), k

def prepare_mpra():
    for root in [Path("/kaggle/input"), OUT, DATA]:
        hits = list(root.rglob("mpra_pairs_meta.csv.gz")) if root.exists() else []
        if hits:
            log(f"MPRA pair table (cached): {hits[0]}"); return pd.read_csv(hits[0], dtype={"chrom": str})
    z = fetch(SIRAJ_ZIP, SIRAJ_URLS, min_bytes=100_000_000, required=False, desc="Siraj et al. 2026 MPRA (Zenodo)")
    if z is None: return None
    ex = DATA / "siraj_counts"; ex.mkdir(exist_ok=True)
    with zipfile.ZipFile(z) as Z:
        names = [n for n in Z.namelist() if n.endswith("_normalized_counts.out.gz") and "/counts/" in n]
        for n in names:
            tgt = ex / Path(n).name
            if not tgt.exists():
                with Z.open(n) as src, open(tgt, "wb") as dst: shutil.copyfileobj(src, dst)
    fs = sorted(ex.glob("*_normalized_counts.out.gz"))
    T = pd.concat(Parallel(n_jobs=min(N_JOBS, 4))(delayed(mpra_eps_table)(str(f)) for f in fs), ignore_index=True)
    T["a"] = T[["x_pos", "y_pos"]].min(axis=1); T["b"] = T[["x_pos", "y_pos"]].max(axis=1)
    rows = []
    for (c, a, b), g in T.groupby(["chrom", "a", "b"]):
        r = dict(chrom=str(c), x_pos=int(g.x_pos.iloc[0]), y_pos=int(g.y_pos.iloc[0]), x_ref=g.x_ref.iloc[0], x_alt=g.x_alt.iloc[0])
        gc_ = g[g.cell_type == CFG["MPRA_CELL"]]
        # orient every measurement to the same (x, y) labelling: eps is symmetric in x<->y, so no sign change is needed
        for lab, sub in (("cell", gc_), ("all", g)):
            if len(sub): m_, s_, p_, k_ = re_meta(sub.eps, sub.eps_se)
            else: m_ = s_ = p_ = np.nan; k_ = 0
            r.update({f"eps_{lab}": m_, f"se_{lab}": s_, f"p_{lab}": p_, f"n_meas_{lab}": k_})
        rows.append(r)
    M = pd.DataFrame(rows)
    for lab in ("cell", "all"):
        ok = M[f"p_{lab}"].notna(); M[f"q_{lab}"] = np.nan
        M.loc[ok, f"q_{lab}"] = multipletests(M.loc[ok, f"p_{lab}"], method="fdr_bh")[1]
    M.to_csv(OUT / "mpra_pairs_meta.csv.gz", index=False)
    log(f"MPRA: {len(T):,} measurements -> {len(M):,} pairs; {CFG['MPRA_CELL']} FDR<5%: {int((M.q_cell < .05).sum())} "
        f"of {int(M.q_cell.notna().sum())}; all cell types FDR<5%: {int((M.q_all < .05).sum())}")
    return M

def lift_hg19_to_hg38(M):
    if not _ensure_pkg("pyliftover"): return None
    from pyliftover import LiftOver
    lo = LiftOver("hg19", "hg38")
    def conv(c, p):
        r = lo.convert_coordinate(f"chr{c}", int(p) - 1)
        return (r[0][1] + 1) if r and r[0][0] == f"chr{c}" else np.nan
    M2 = M.copy(); M2["x_pos"] = [conv(c, p) for c, p in zip(M.chrom, M.x_pos)]; M2["y_pos"] = [conv(c, p) for c, p in zip(M.chrom, M.y_pos)]
    return M2.dropna(subset=["x_pos", "y_pos"]).astype({"x_pos": int, "y_pos": int})

def _ensure_pkg(pkg):
    try: __import__(pkg); return True
    except ImportError:
        try: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True, timeout=600); __import__(pkg); return True
        except Exception as e: log(f"could not install {pkg}: {e}"); return False

def targeted_gene(g, Hg, a_all, b_all, half, jobs):
    """jobs: list of (pair_id, jx, jy). Pre-specified pairs -> tested in ALL individuals and in each half."""
    ok = (a_all + b_all) >= CFG["MIN_COUNT"]
    hA = Hg[:, :, 0].astype(float); hB = Hg[:, :, 1].astype(float); dA = hA - hB
    L = emp_logit(a_all[ok], b_all[ok]); w = emp_w(a_all[ok], b_all[ok])
    D = (hA + hB)[ok]; sd = D.std(0); okv = sd > 1e-9
    Zs = np.zeros_like(D); Zs[:, okv] = (D[:, okv] - D[:, okv].mean(0)) / sd[okv]
    r = np.nan_to_num(wcorr_cols(dA[ok], L, w)); r[~okv] = 0
    out = []
    for pid, jx, jy in jobs:
        rec = dict(pair_id=pid, gene=GENES.index[g], gene_name=GENES.gene_name.iloc[g])
        if not (okv[jx] and okv[jy]): rec["status"] = "monomorphic"; out.append(rec); continue
        r2 = float((Zs[:, jx] @ Zs[:, jy] / len(Zs)) ** 2); rec["r2_ld"] = r2
        if r2 > CFG["TARGET_MAX_R2"]: rec["status"] = "LD too high"; out.append(rec); continue
        rr = r.copy(); rr[[jx, jy]] = 0
        cond = clump(Zs, rr, okv, CFG["N_COND"], CFG["CLUMP_R2"])
        t = pair_test(hA[ok], hB[ok], a_all[ok], b_all[ok], jx, jy, cond)
        if t is None: rec["status"] = "too few informative haplotypes"; out.append(rec); continue
        rec.update(status="tested", **t)
        for lab, hv in (("disc", 0), ("rep", 1)):
            m = ok & (half == hv)
            th = pair_test(hA[m], hB[m], a_all[m], b_all[m], jx, jy, cond)
            if th: rec[f"{lab}_gamma"], rec[f"{lab}_se"], rec[f"{lab}_p"] = th["gamma"], th["gamma_se"], th["p"]
        out.append(rec)
    return out

if CFG["MODE"] == "BATCH" and CFG["USE_MPRA_TARGETED"]:
    t_m = time.time()
    MP = prepare_mpra()
    TGT = pd.DataFrame()
    if MP is not None and len(MP):
        MP["chrom"] = MP.chrom.astype(str).str.replace("chr", "")
        bch = {c.replace("chr", "") for c in CHROMS}
        snv = {(c.replace("chr", ""), p): i for i, (c, p, rf, al) in enumerate(zip(VAR.chrom, VAR.pos, VAR.ref, VAR.alt))}
        snv_full = {(c.replace("chr", ""), p, rf, al) for c, p, rf, al in zip(VAR.chrom, VAR.pos, VAR.ref, VAR.alt)}
        def rate(M):
            sub = M[M.chrom.isin(bch)]
            return (np.mean([(c, p, r_, a_) in snv_full for c, p, r_, a_ in zip(sub.chrom, sub.x_pos, sub.x_ref, sub.x_alt)])
                    if len(sub) else np.nan), len(sub)
        r38, nb = rate(MP); build = "GRCh38 (as provided)"
        log(f"MPRA pairs on this batch: {nb}; focal-variant exact match to the GRCh38 VCF: {r38:.1%}")
        if nb and (np.isnan(r38) or r38 < 0.3):
            M2 = lift_hg19_to_hg38(MP)
            if M2 is not None:
                r19, _ = rate(M2)
                log(f"  after hg19->hg38 liftover: {r19:.1%}")
                if r19 > r38: MP, build = M2, "hg19 lifted to GRCh38"
        MPb = MP[MP.chrom.isin(bch)].copy(); MPb["pair_id"] = MPb.chrom + ":" + MPb.x_pos.astype(str) + "~" + MPb.y_pos.astype(str)
        jobs = defaultdict(list); n_both = 0
        tss = GENES.tss.values; gch = GENES.chrom.str.replace("chr", "").values
        for _, q in MPb.iterrows():
            ix, iy = snv.get((q.chrom, int(q.x_pos))), snv.get((q.chrom, int(q.y_pos)))
            if ix is None or iy is None: continue
            n_both += 1
            for g in np.where((gch == q.chrom) & (np.abs(tss - q.x_pos) <= CFG["CIS_WINDOW"]))[0]:
                loc = {v: k for k, v in enumerate(GVARS[g])}
                if ix in loc and iy in loc: jobs[g].append((q.pair_id, loc[ix], loc[iy]))
        log(f"MPRA pairs with both variants common (MAF>={CFG['MAF_MIN']}) biallelic SNVs in 1000G: {n_both}; "
            f"pair x gene tests to run: {sum(len(v) for v in jobs.values())} ({build})")
        res = Parallel(n_jobs=N_JOBS, backend="loky", verbose=0)(
            delayed(targeted_gene)(g, np.ascontiguousarray(HAP[GVARS[g]][:, S_V, :].transpose(1, 0, 2)), GA[g], GB[g], HALF, jb)
            for g, jb in jobs.items())
        TGT = pd.DataFrame([r for rr in res for r in rr])
        if len(TGT):
            TGT = TGT.merge(MPb.drop(columns=["chrom"]), on="pair_id", how="left"); TGT["build"] = build
            TGT["n_mpra_pairs_batch"] = nb; TGT["n_mpra_pairs_common"] = n_both
        log(f"Targeted MPRA-pair tests: {TGT.status.value_counts().to_dict() if len(TGT) else 'none'} "
            f"in {(time.time()-t_m)/60:.1f} min")
    tag = f"batch{CFG['BATCH']}" if not CFG["CHROMS_OVERRIDE"] else "pilot_" + "_".join(CHROMS)
    TGT.to_csv(OUT / f"mpra_targeted_{tag}.csv.gz", index=False)

## 7. COMBINE — genome-wide FDR, replication, evidence levels, controls, figures (also runs at the end of a pilot)

In [ ]:
def read_csv_safe(f):
    try: return pd.read_csv(f)
    except Exception: return pd.DataFrame()

def find_outputs(prefix):
    hits = []
    for root in [Path("/kaggle/input"), OUT]:
        if root.exists(): hits += list(root.rglob(f"{prefix}_*.csv.gz"))
    uniq = {}
    for h in hits: uniq.setdefault(h.name, h)
    return sorted(uniq.values())

def lambda_gc(p):
    p = np.asarray(p, float); p = p[~np.isnan(p)]
    return float(np.median(chi2.isf(p, 1)) / chi2.ppf(0.5, 1)) if len(p) else np.nan

def boot_ci(x, blocks, B):
    rng = np.random.default_rng(0); x = np.asarray(x); ok = ~np.isnan(x); x, blocks = x[ok], np.asarray(blocks)[ok]
    ub = np.unique(blocks); idx = [np.where(blocks == u)[0] for u in ub]
    bs = [x[np.concatenate([idx[j] for j in rng.integers(0, len(ub), len(ub))])].mean() for _ in range(B)]
    return tuple(np.percentile(bs, [2.5, 97.5]))

run_combine = CFG["MODE"] == "COMBINE" or (CFG["MODE"] == "BATCH" and (CFG["CHROMS_OVERRIDE"] or CFG["SELF_TEST"]))
if run_combine:
    fs = find_outputs("ase_scan"); fp = find_outputs("ase_pred"); fg = find_outputs("geu_rep")
    log("COMBINE inputs:", [f.name for f in fs])
    if not fs: raise SystemExit("No ase_scan_*.csv.gz found. Attach the BATCH outputs as inputs.")
    ALL = pd.concat([read_csv_safe(f) for f in fs], ignore_index=True).drop_duplicates(["gene", "kind", "v1", "v2"])
    PR = pd.concat([read_csv_safe(f) for f in fp], ignore_index=True) if fp else pd.DataFrame()
    if len(PR): PR = PR.drop_duplicates("gene")
    GR = pd.concat([read_csv_safe(f) for f in fg], ignore_index=True) if fg else pd.DataFrame()
    if len(GR): GR = GR.drop_duplicates(["gene", "v1", "v2"])
    REAL = ALL[ALL.kind == "real"].copy()
    if len(GR): REAL = REAL.merge(GR, on=["gene", "v1", "v2"], how="left")
    REAL = REAL.reset_index(drop=True)
    REAL["disc_q"] = multipletests(REAL.disc_p.fillna(1), method="fdr_bh")[1]
    z = REAL.rep_gamma / REAL.rep_gamma_se
    REAL["rep_p1"] = np.where(np.sign(REAL.rep_gamma) == np.sign(REAL.disc_gamma), norm.sf(np.abs(z)), 1 - norm.sf(np.abs(z)))
    sig = REAL.disc_q < CFG["FDR"]
    REAL["rep_q"] = np.nan
    if sig.any(): REAL.loc[sig, "rep_q"] = multipletests(REAL.loc[sig, "rep_p1"].fillna(1), method="fdr_bh")[1]
    w1, w2 = 1 / REAL.disc_gamma_se ** 2, 1 / REAL.rep_gamma_se ** 2
    REAL["meta_gamma"] = (REAL.disc_gamma * w1 + REAL.rep_gamma * w2) / (w1 + w2); REAL["meta_se"] = 1 / np.sqrt(w1 + w2)
    REAL["meta_p"] = 2 * norm.sf(np.abs(REAL.meta_gamma / REAL.meta_se))
    b1, b2, gm = REAL.disc_beta1, REAL.disc_beta2, REAL.disc_gamma; joint = b1 + b2 + gm
    REAL["pattern"] = np.select(
        [(b1 * b2 < 0) & (np.abs(joint) < np.maximum(np.abs(b1), np.abs(b2))),
         (b1 * b2 > 0) & (np.sign(gm) != np.sign(b1 + b2)) & (np.abs(joint) < np.abs(b1 + b2)),
         (np.sign(gm) == np.sign(b1 + b2)) & (np.abs(joint) > np.abs(b1 + b2))],
        ["compensatory (opposing effects, joint near reference)", "buffering (sub-additive)", "synergistic (super-additive)"], "other")
    rep_ok = sig & (REAL.rep_p1 < CFG["REP_P"])
    REAL["evidence_level"] = 1
    REAL.loc[sig, "evidence_level"] = 2
    REAL.loc[rep_ok, "evidence_level"] = 3
    both_ccre = REAL.get("v1_ccre", False).fillna(False).astype(bool) & REAL.get("v2_ccre", False).fillna(False).astype(bool) \
        if "v1_ccre" in REAL else pd.Series(False, index=REAL.index)
    REAL.loc[rep_ok & both_ccre, "evidence_level"] = 4
    REAL["geu_support"] = REAL.get("geu_p1", pd.Series(np.nan, index=REAL.index)) < 0.05

    # ---- experimental evidence supplied by the user (wet-lab results attached as Kaggle input) ----
    #   MPRA table   : columns chrom,pos1,pos2 + (y00,y10,y01,y11 [,se]) or (eps [,eps_se])   -> L4 (experimental support)
    #   Perturbation : columns chrom,pos1,pos2,perturb_effect,perturb_p  (CRISPR / base editing) -> L5 (mechanistic)
    def _key(c, p1, p2):
        c = str(c).replace("chr", ""); a, b = sorted((int(p1), int(p2))); return (c, a, b)
    REAL["_key"] = [_key(v1.split(":")[0], v1.split(":")[1], v2.split(":")[1]) for v1, v2 in zip(REAL.v1, REAL.v2)]
    EXP = {"mpra": {}, "perturb": {}}
    roots = [Path("/kaggle/input"), DATA / "experimental"]
    for root in roots:
        if not root.exists(): continue
        for f in root.rglob("*"):
            if not f.is_file() or f.suffix.lower() not in (".csv", ".tsv", ".txt", ".gz", ".xlsx") or f.stat().st_size > 5e8: continue
            if f.name.startswith(("ase_", "geu_rep_", "meta_")): continue
            try:
                df = pd.read_excel(f) if f.suffix.lower() == ".xlsx" else pd.read_csv(f, sep=None, engine="python")
            except Exception:
                continue
            low = {c.lower(): c for c in df.columns}
            if not all(k in low for k in ("chrom", "pos1", "pos2")): continue
            if "perturb_effect" in low and "perturb_p" in low:
                for c, p1, p2, e, p in zip(df[low["chrom"]], df[low["pos1"]], df[low["pos2"]], df[low["perturb_effect"]], df[low["perturb_p"]]):
                    EXP["perturb"][_key(c, p1, p2)] = (float(e), float(p))
                log(f"Perturbation results loaded from {f.name}: {len(df)} pairs")
            elif all(k in low for k in ("y00", "y10", "y01", "y11")) or "eps" in low:
                if "eps" in low: eps = df[low["eps"]].astype(float).values
                else: eps = (df[low["y11"]] - df[low["y10"]] - df[low["y01"]] + df[low["y00"]]).astype(float).values
                se_col = low.get("eps_se") or low.get("se")
                pv = 2 * norm.sf(np.abs(eps / df[se_col].astype(float).values)) if se_col else np.full(len(eps), np.nan)
                for c, p1, p2, e, p in zip(df[low["chrom"]], df[low["pos1"]], df[low["pos2"]], eps, pv):
                    EXP["mpra"][_key(c, p1, p2)] = (float(e), float(p))
                log(f"MPRA results loaded from {f.name}: {len(df)} pairs")
    def _support(key, gam, kind):
        if key not in EXP[kind]: return np.nan
        e, p = EXP[kind][key]
        return float(np.sign(e) == np.sign(gam) and (np.isnan(p) or p < 0.05))
    REAL["mpra_support"] = [_support(k, g_, "mpra") for k, g_ in zip(REAL._key, REAL.disc_gamma)]
    REAL["perturb_support"] = [_support(k, g_, "perturb") for k, g_ in zip(REAL._key, REAL.disc_gamma)]
    REAL.loc[rep_ok & (REAL.mpra_support == 1), "evidence_level"] = 4
    REAL.loc[rep_ok & (REAL.perturb_support == 1), "evidence_level"] = 5
    REAL["evidence_label"] = REAL.evidence_level.map({
        1: "L1 tested", 2: "L2 statistical interaction (discovery FDR)", 3: "L3 replicated (independent individuals)",
        4: "L4 replicated + functional support", 5: "L5 replicated + perturbation-validated"})
    REAL.loc[(REAL.evidence_level == 4) & (REAL.mpra_support != 1), "evidence_label"] = "L4* replicated + ENCODE cCRE annotation (not experimental)"
    n_exp = len(EXP["mpra"]) + len(EXP["perturb"])
    log(f"Experimental evidence: {len(EXP['mpra'])} MPRA pairs, {len(EXP['perturb'])} perturbation pairs attached"
        + ("" if n_exp else " (none attached: L4 = cCRE annotation only, no L5)"))
    REAL = REAL.drop(columns=["_key"])
    REAL.to_csv(OUT / "ase_interactions_genomewide.csv.gz", index=False)

    # ---- calibration & negative controls ----
    perm = ALL[ALL.kind.str.startswith("perm")].disc_p; rnd = ALL[ALL.kind == "random"].disc_p
    CTRL = pd.DataFrame([
        dict(test="real cis-epistasis (discovery)", n=len(REAL), lambda_gc=lambda_gc(REAL.disc_p),
             n_fdr05=int(sig.sum()), frac_p05=float((REAL.disc_p < .05).mean())),
        dict(test="orientation-flip permutation null", n=len(perm), lambda_gc=lambda_gc(perm),
             n_fdr05=int((multipletests(perm.fillna(1), method='fdr_bh')[1] < CFG["FDR"]).sum()) if len(perm) else 0,
             frac_p05=float((perm < .05).mean()) if len(perm) else np.nan),
        dict(test="MAF/LD-matched random pairs", n=len(rnd), lambda_gc=lambda_gc(rnd),
             n_fdr05=int((multipletests(rnd.fillna(1), method='fdr_bh')[1] < CFG["FDR"]).sum()) if len(rnd) else 0,
             frac_p05=float((rnd < .05).mean()) if len(rnd) else np.nan)])
    if len(REAL):   # replication rate among all discovery p<1e-3 (pi1-style)
        top = REAL[REAL.disc_p < 1e-3]
        CTRL["replication_sign_concordance_top"] = float((np.sign(top.rep_gamma) == np.sign(top.disc_gamma)).mean()) if len(top) else np.nan
    save_table(CTRL, "V2_S1_calibration_controls"); print(CTRL.to_string())

    # ---- compensation enrichment among replicated vs all tested ----
    ENR = []
    for lab, m in [("L2 discovery FDR", REAL.evidence_level >= 2), ("L3 replicated", REAL.evidence_level >= 3)]:
        if m.sum() == 0: continue
        cmp = REAL.pattern.str.startswith("compensatory")
        a_, b_ = int((m & cmp).sum()), int((m & ~cmp).sum()); c_, d_ = int((~m & cmp).sum()), int((~m & ~cmp).sum())
        orr, p = fisher_exact([[a_, b_], [c_, d_]])
        ENR.append(dict(set=lab, n=a_ + b_, frac_compensatory=a_ / max(a_ + b_, 1), background=c_ / max(c_ + d_, 1), OR=orr, p=p))
        if "v1_ccre" in REAL:
            cc = REAL.v1_ccre.fillna(False).astype(bool) | REAL.v2_ccre.fillna(False).astype(bool)
            a_, b_ = int((m & cc).sum()), int((m & ~cc).sum()); c_, d_ = int((~m & cc).sum()), int((~m & ~cc).sum())
            orr, p = fisher_exact([[a_, b_], [c_, d_]])
            ENR.append(dict(set=lab + " — cCRE overlap", n=a_ + b_, frac_compensatory=a_ / max(a_ + b_, 1),
                            background=c_ / max(c_ + d_, 1), OR=orr, p=p))
    ENR = save_table(pd.DataFrame(ENR), "V2_T2_compensation_and_ccre_enrichment")

    # ---- prediction summary ----
    PSUM = pd.DataFrame()
    if len(PR):
        blocks = PR.chrom.astype(str) + "_" + (PR.tss // 2_000_000).astype(str)
        rows = []
        for k in ["additive", "additive+cis", "additive+cis(shuffled phase)"]:
            d = PR[f"R2_{k}"] - PR["R2_additive"]
            lo, hi = boot_ci(d.values, blocks.values, CFG["BOOTSTRAP"]) if k != "additive" else (0, 0)
            try: wp = wilcoxon(PR[f"R2_{k}"], PR["R2_additive"], nan_policy="omit").pvalue if k != "additive" else np.nan
            except Exception: wp = np.nan
            rows.append(dict(model=k, genes=int(PR[f"R2_{k}"].notna().sum()), R2_mean=PR[f"R2_{k}"].mean(),
                             R2_median=PR[f"R2_{k}"].median(), dR2_vs_additive=np.nanmean(d), dR2_CI=f"[{lo:.5f}, {hi:.5f}]",
                             wilcoxon_p=wp, genes_with_cis_terms=int((PR.n_cis_terms_selected > 0).sum()) if k == "additive+cis" else np.nan))
        PSUM = save_table(pd.DataFrame(rows), "V2_T1_heldout_allelic_prediction"); print(PSUM.to_string())

    # ---- candidate table ----
    cols = ["gene", "gene_name", "chrom", "v1", "v2", "r2_ld", "disc_n_inf", "disc_beta1", "disc_beta2", "disc_gamma",
            "disc_gamma_se", "disc_p", "disc_q", "rep_gamma", "rep_p1", "rep_q", "meta_gamma", "meta_p", "pattern",
            "v1_ccre", "v2_ccre", "geu_n", "geu_cis_minus_trans", "geu_p1", "mpra_support", "perturb_support",
            "evidence_level", "evidence_label"]
    T = REAL.sort_values(["evidence_level", "disc_p"], ascending=[False, True])[[c for c in cols if c in REAL]].head(100)
    save_table(T, "V2_T3_candidate_cis_epistasis_compensation")
    print("Evidence levels:", REAL.evidence_level.value_counts().sort_index().to_dict())
    print(T.head(15).to_string())

    # ---- cross-ancestry note & v1 integration ----
    V1 = {}
    for nm in ["T3_main_expression_prediction_baselines.csv", "T5_ablations_A0_A8.csv"]:
        hits = list(Path("/kaggle/input").rglob(nm)) if Path("/kaggle/input").exists() else []
        if hits: V1[nm] = pd.read_csv(hits[0])
    if V1: log("v1 total-expression results found and included in the summary:", list(V1))

    # ---- figures ----
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
    def qq(ax, p, lab, c):
        p = np.sort(np.asarray(p, float)[~np.isnan(p)])
        if len(p) == 0: return
        e = (np.arange(1, len(p) + 1) - 0.5) / len(p)
        ax.plot(-np.log10(e), -np.log10(np.clip(p, 1e-300, 1)), ".", ms=2.5, c=c, label=f"{lab} (λ={lambda_gc(p):.2f}, n={len(p):,})")
    qq(axs[0], REAL.disc_p, "cis-epistasis, real", PAL[1]); qq(axs[0], perm, "orientation-flip null", "#888")
    qq(axs[0], rnd, "random pairs", PAL[0])
    m_ = axs[0].get_xlim()[1]; axs[0].plot([0, m_], [0, m_], c="k", lw=.7); axs[0].legend(frameon=False, fontsize=7)
    axs[0].set_xlabel("expected −log10 p"); axs[0].set_ylabel("observed −log10 p"); axs[0].set_title("a  Calibration")
    v = REAL.dropna(subset=["rep_gamma"])
    axs[1].scatter(v.disc_gamma, v.rep_gamma, s=4, alpha=.35, c="#999")
    vv = v[v.evidence_level >= 2]; axs[1].scatter(vv.disc_gamma, vv.rep_gamma, s=12, c=PAL[1], label="discovery FDR<5%")
    axs[1].axhline(0, c="#ccc"); axs[1].axvline(0, c="#ccc"); axs[1].legend(frameon=False, fontsize=7)
    axs[1].set_xlabel("γ discovery half"); axs[1].set_ylabel("γ replication half"); axs[1].set_title("b  Independent replication")
    if len(PR):
        axs[2].hist((PR["R2_additive+cis"] - PR["R2_additive"]).dropna().clip(-.1, .1), 60, color=PAL[3], alpha=.8, label="true phase")
        axs[2].hist((PR["R2_additive+cis(shuffled phase)"] - PR["R2_additive"]).dropna().clip(-.1, .1), 60, color="#999", alpha=.6, label="shuffled phase")
        axs[2].axvline(0, c="k", lw=.7); axs[2].legend(frameon=False, fontsize=7)
        axs[2].set_xlabel("ΔR² (allelic log-ratio) vs additive"); axs[2].set_title("c  Held-out prediction")
    fig.suptitle("Figure V2-1 | Haplotype-resolved cis-epistasis (MAGE ASE)", x=.01, ha="left", weight="bold"); savefig(fig, "FigV2_1_overview")

    fp_ = REAL[REAL.evidence_level >= 3].sort_values("meta_p").head(20)
    if len(fp_):
        fig, ax = plt.subplots(figsize=(7, .35 * len(fp_) + 1.4)); yy = np.arange(len(fp_))
        ax.errorbar(fp_.disc_gamma, yy - .15, xerr=1.96 * fp_.disc_gamma_se, fmt="o", c=PAL[0], label="discovery")
        ax.errorbar(fp_.rep_gamma, yy + .15, xerr=1.96 * fp_.rep_gamma_se, fmt="s", c=PAL[2], label="replication")
        ax.set_yticks(yy); ax.set_yticklabels([f"{g} {a.split(':')[1]}×{b.split(':')[1]} ({p.split(' ')[0]})"
                                               for g, a, b, p in zip(fp_.gene_name, fp_.v1, fp_.v2, fp_.pattern)], fontsize=7)
        ax.axvline(0, c="#999"); ax.invert_yaxis(); ax.legend(frameon=False); ax.set_xlabel("γ (log allelic-ratio scale, 95% CI)")
        ax.set_title("Figure V2-2 | Replicated cis-epistatic pairs"); savefig(fig, "FigV2_2_replicated_forest")
        q = fp_.iloc[0]
        fig, ax = plt.subplots(figsize=(4.2, 3.2))
        hap = [0, q.disc_beta1, q.disc_beta2, q.disc_beta1 + q.disc_beta2 + q.disc_gamma]
        add = [0, q.disc_beta1, q.disc_beta2, q.disc_beta1 + q.disc_beta2]
        ax.plot(range(4), hap, "o-", c=PAL[1], label="fitted haplotype effect"); ax.plot(range(4), add, "s--", c="#888", label="additive expectation")
        ax.set_xticks(range(4)); ax.set_xticklabels(["ref", "v1 alt", "v2 alt", "both alt\n(same copy)"])
        ax.axhline(0, c="#ccc"); ax.set_ylabel("log haplotype expression (vs ref)"); ax.legend(frameon=False, fontsize=7)
        ax.set_title(f"Figure V2-3 | {q.gene_name}: {q.pattern.split(' (')[0]}"); savefig(fig, "FigV2_3_top_haplotype_effects")
    if len(ENR):
        fig, ax = plt.subplots(figsize=(6, 3.2)); x = np.arange(len(ENR))
        ax.bar(x - .2, ENR.frac_compensatory, .4, color=PAL[1], label="candidates"); ax.bar(x + .2, ENR.background, .4, color="#aaa", label="all tested")
        ax.set_xticks(x); ax.set_xticklabels([f"{s}\np={p:.2g}" for s, p in zip(ENR.set, ENR.p)], fontsize=7)
        ax.legend(frameon=False); ax.set_title("Figure V2-4 | Compensation & cCRE enrichment"); savefig(fig, "FigV2_4_enrichment")

    # ---------------- Gap A: MPRA-pair transfer to natural haplotypes ----------------
    GAPA = {}
    ft = find_outputs("mpra_targeted")
    TG = pd.concat([read_csv_safe(f) for f in ft], ignore_index=True) if ft else pd.DataFrame()
    if len(TG) and "status" in TG:
        TG = TG.drop_duplicates(["pair_id", "gene"])
        tested = TG[TG.status == "tested"].copy()
        GAPA["status_counts"] = TG.status.value_counts().to_dict()
        GAPA["n_pairs_common_in_1000G"] = int(TG.drop_duplicates("pair_id").shape[0])
        if len(tested):
            tested["q_nat"] = multipletests(tested.p.fillna(1), method="fdr_bh")[1]
            # one row per pair: gene with the smallest p, Bonferroni-corrected over the genes tested for that pair
            ng = tested.groupby("pair_id").gene.transform("count")
            tested["p_pair"] = np.minimum(tested.p * ng, 1)
            PP = tested.sort_values("p").drop_duplicates("pair_id").copy()
            PP["q_pair"] = multipletests(PP.p_pair.fillna(1), method="fdr_bh")[1]
            GAPA.update(n_pairs_tested=len(PP), n_nat_fdr=int((PP.q_pair < CFG["FDR"]).sum()))
            def gapa_stats(sc):
                st = {}; e_, q_, p_ = PP[f"eps_{sc}"], PP[f"q_{sc}"], PP[f"p_{sc}"]
                sig_, null_ = q_ < CFG["FDR"], p_ > 0.5
                st["n_mpra_sig_tested"] = int(sig_.sum()); st["n_mpra_null_tested"] = int(null_.sum())
                a_ = int((sig_ & (PP.p < .05)).sum()); b_ = int((sig_ & (PP.p >= .05)).sum())
                c_ = int((null_ & (PP.p < .05)).sum()); d_ = int((null_ & (PP.p >= .05)).sum())
                if a_ + b_ and c_ + d_:
                    orr, pf = fisher_exact([[a_, b_], [c_, d_]], alternative="greater")
                    st.update(frac_nat_p05_mpra_sig=a_ / (a_ + b_), frac_nat_p05_mpra_null=c_ / (c_ + d_), fisher_OR=orr, fisher_p=pf)
                cs = PP[sig_ & e_.notna()]
                if len(cs):
                    agree = int((np.sign(cs[f"eps_{sc}"]) == np.sign(cs.gamma)).sum())
                    st.update(sign_concordance=agree / len(cs), sign_binom_p=stats.binomtest(agree, len(cs), 0.5, alternative="greater").pvalue)
                    cs2 = cs[cs.p < .05]
                    if len(cs2):
                        ag2 = int((np.sign(cs2[f"eps_{sc}"]) == np.sign(cs2.gamma)).sum())
                        st.update(sign_concordance_nat_p05=ag2 / len(cs2), n_nat_p05=len(cs2),
                                  sign_binom_p_nat_p05=stats.binomtest(ag2, len(cs2), 0.5, alternative="greater").pvalue)
                ok2 = e_.notna() & PP.gamma.notna()
                if ok2.sum() >= 10:
                    sp = spearmanr(e_[ok2], PP.gamma[ok2]); st.update(spearman_eps_vs_gamma=sp.correlation, spearman_p=sp.pvalue)
                return st
            PRIM = CFG["MPRA_PRIMARY"]; SEC = "cell" if PRIM == "all" else "all"
            GAPA["primary_scope"] = "all cell types" if PRIM == "all" else CFG["MPRA_CELL"]
            GAPA["primary"] = gapa_stats(PRIM); GAPA["secondary"] = gapa_stats(SEC)
            mpra_sig = PP[f"q_{PRIM}"] < CFG["FDR"]; mpra_null = PP[f"p_{PRIM}"] > 0.5
            PP["eps_primary"] = PP[f"eps_{PRIM}"]
            ok_ = PP.eps_primary.notna() & PP.gamma.notna()
            if "rep_gamma" in PP and "disc_gamma" in PP:
                rr_ = PP.dropna(subset=["disc_gamma", "rep_gamma"]); f_ = rr_.q_pair < CFG["FDR"]
                GAPA["half_sign_concordance_nat_fdr"] = float((np.sign(rr_[f_].disc_gamma) == np.sign(rr_[f_].rep_gamma)).mean()) if f_.any() else np.nan
            PP["gapA_evidence"] = np.select(
                [(PP.q_pair < CFG["FDR"]) & mpra_sig & (np.sign(PP.eps_primary) == np.sign(PP.gamma)),
                 (PP.q_pair < CFG["FDR"]), mpra_sig & (PP.p < .05) & (np.sign(PP.eps_primary) == np.sign(PP.gamma))],
                ["natural FDR + MPRA FDR, same sign (experimental + population support)", "natural FDR only",
                 "MPRA FDR + natural nominal, same sign"], "")
            keepc = [c for c in ["pair_id", "gene", "gene_name", "x_pos", "y_pos", "x_ref", "x_alt", "r2_ld", "n_inf", "n",
                                 "beta1", "beta2", "gamma", "gamma_se", "p", "p_pair", "q_pair", "disc_gamma", "rep_gamma",
                                 "eps_all", "q_all", "eps_cell", "q_cell", "gapA_evidence", "build"] if c in PP]
            save_table(PP.sort_values("p")[keepc].head(200), "V2_T4_MPRA_pairs_on_natural_haplotypes")
            PP.to_csv(OUT / "mpra_targeted_all_pairs.csv.gz", index=False)
            fig, axs = plt.subplots(1, 2, figsize=(11, 4))
            v = PP[ok_]
            axs[0].scatter(v.eps_primary, v.gamma, s=6, c=np.where(v[f"q_{PRIM}"] < CFG["FDR"], PAL[1], "#bbb"), alpha=.6)
            axs[0].axhline(0, c="#ccc"); axs[0].axvline(0, c="#ccc")
            axs[0].set_xlabel(f"MPRA ε ({GAPA['primary_scope']}, log2 activity)"); axs[0].set_ylabel("natural-haplotype γ (log allelic ratio)")
            axs[0].set_title(f"a  ρ={GAPA['primary'].get('spearman_eps_vs_gamma', np.nan):.2f} (red: MPRA FDR<5%)")
            for lab, m, c in (("MPRA FDR<5%", mpra_sig, PAL[1]), ("MPRA additive (p>0.5)", mpra_null, "#888")):
                p = np.sort(PP.p[m].dropna().values)
                if len(p): axs[1].plot(-np.log10((np.arange(1, len(p) + 1) - .5) / len(p)), -np.log10(np.clip(p, 1e-300, 1)), ".", ms=3, c=c, label=f"{lab} (n={len(p)})")
            mm = axs[1].get_xlim()[1]; axs[1].plot([0, mm], [0, mm], c="k", lw=.7); axs[1].legend(frameon=False, fontsize=7)
            axs[1].set_xlabel("expected −log10 p"); axs[1].set_ylabel("observed −log10 p (natural γ)"); axs[1].set_title("b  Natural-haplotype test by MPRA status")
            fig.suptitle("Figure V2-5 | Gap A: do MPRA-measured non-additive pairs act non-additively on natural haplotypes?", x=.01, ha="left", weight="bold")
            savefig(fig, "FigV2_5_gapA_mpra_transfer")
        save_table(pd.json_normalize(GAPA), "V2_S2_gapA_summary"); print("Gap A:", json.dumps(GAPA, default=str, indent=1))
    else:
        log("Gap A: no mpra_targeted_* outputs found (USE_MPRA_TARGETED off, or MPRA download failed).")

    SUMMARY = dict(n_tests=len(REAL), n_genes=int(REAL.gene.nunique()), evidence=REAL.evidence_level.value_counts().to_dict(),
                   controls=CTRL.to_dict("records"), prediction=PSUM.to_dict("records") if len(PSUM) else None,
                   enrichment=ENR.to_dict("records") if len(ENR) else None, v1_included=list(V1), gap_A=GAPA,
                   config_hash=cfg_hash(), self_test=CFG["SELF_TEST"])
    with open(OUT / "v2_results_summary.json", "w") as f: json.dump(SUMMARY, f, indent=1, default=str)
    with open(OUT / "run_log.txt", "w") as f: f.write("\n".join(LOG))
    zp = shutil.make_archive(str(WORK / "originsib_v2_results"), "zip", OUT)
    log(f"COMBINE done. Results: {OUT} | zip: {zp}")
elif CFG["MODE"] == "BATCH":
    with open(OUT / f"run_log_batch{CFG['BATCH']}.txt", "w") as f: f.write("\n".join(LOG))
    zp = shutil.make_archive(str(WORK / f"originsib_v2_batch{CFG['BATCH']}"), "zip", OUT)
    log(f"BATCH {CFG['BATCH']} done in {(time.time()-T0)/60:.1f} min. Zip: {zp}. Run the next batch, then COMBINE.")